## Get dataset E. Coli

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/genome-firewall")
RAW_DIR = PROJECT / "data" / "raw"
PROCESSED_DIR = PROJECT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT)

Mounted at /content/drive
/content/drive/MyDrive/genome-firewall


In [2]:
!pip -q install requests pandas pyarrow

In [3]:
import io
import time
import requests
import pandas as pd
from pathlib import Path
from urllib.parse import quote

In [4]:
API_BASE = "https://www.bv-brc.org/api"
AMR_ENDPOINT = f"{API_BASE}/genome_amr/"

In [5]:
test_url = (
    f"{AMR_ENDPOINT}?"
    "limit(5)"
    "&http_accept=application/json"
)

response = requests.get(test_url, timeout=60)
response.raise_for_status()

test_records = response.json()

print("Records returned:", len(test_records))
test_records[:2]

Records returned: 5


[{'id': '3fb4c919-78a5-4ef4-891f-a806c6a90b7b',
  'owner': 'PATRIC@patricbrc.org',
  'genome_id': '573.23123',
  'genome_name': 'Klebsiella pneumoniae strain PN84',
  'antibiotic': 'imipenem',
  'computational_method_performance': 'Accuracy:0.931, F1 score:0.935, AUC:0.950',
  'date_inserted': '2019-01-30T02:23:33.633Z',
  'resistant_phenotype': 'Susceptible',
  'public': True,
  'taxon_id': 573,
  'date_modified': '2019-01-30T02:23:33.633Z',
  'computational_method': 'AdaBoost Classifier',
  'evidence': 'Computational Method',
  '_version_': 1818811210820223000},
 {'evidence': 'Laboratory Method',
  'testing_standard_year': 2021,
  'taxon_id': 28901,
  'measurement_value': '4',
  'date_modified': '2024-03-19T13:42:46.004Z',
  'resistant_phenotype': 'Susceptible',
  'public': True,
  'date_inserted': '2024-03-19T13:42:46.004Z',
  'laboratory_typing_method': 'MIC',
  'measurement': '<4',
  'antibiotic': 'azithromycin',
  'measurement_sign': '<',
  'genome_id': '28901.24388',
  'genome_n

In [6]:
if not test_records:
    raise RuntimeError("The API returned no records.")

available_fields = sorted(test_records[0].keys())

for field in available_fields:
    print(field)

_version_
antibiotic
computational_method
computational_method_performance
date_inserted
date_modified
evidence
genome_id
genome_name
id
owner
public
resistant_phenotype
taxon_id


In [7]:
import requests
import pandas as pd

API_BASE = "https://www.bv-brc.org/api"
AMR_ENDPOINT = f"{API_BASE}/genome_amr/"

lab_test_url = (
    f"{AMR_ENDPOINT}?"
    "eq(evidence,Laboratory Method)"
    "&limit(100)"
    "&http_accept=application/json"
)

response = requests.get(lab_test_url, timeout=60)
response.raise_for_status()

lab_records = response.json()

print("Laboratory records returned:", len(lab_records))
lab_records[:2]

Laboratory records returned: 100


[{'evidence': 'Laboratory Method',
  'testing_standard_year': 2021,
  'taxon_id': 28901,
  'measurement_value': '4',
  'date_modified': '2024-03-19T13:42:46.004Z',
  'resistant_phenotype': 'Susceptible',
  'public': True,
  'date_inserted': '2024-03-19T13:42:46.004Z',
  'laboratory_typing_method': 'MIC',
  'measurement': '<4',
  'antibiotic': 'azithromycin',
  'measurement_sign': '<',
  'genome_id': '28901.24388',
  'genome_name': 'Salmonella enterica A038_2016',
  'owner': 'PATRIC@patricbrc.org',
  'testing_standard': 'CLSI',
  'pmid': [35651495],
  'measurement_unit': 'mg/L',
  'id': '3fb4cc8b-497a-43f7-b6de-7e723ba378a3',
  '_version_': 1818811210821271600},
 {'measurement': '',
  'date_inserted': '2020-04-02T18:07:07.838Z',
  'laboratory_typing_method': 'Disk diffusion',
  'taxon_id': 562,
  'measurement_value': '',
  'date_modified': '2020-04-02T18:07:07.838Z',
  'evidence': 'Laboratory Method',
  'resistant_phenotype': 'Resistant',
  'public': True,
  'vendor': 'Oxoid',
  'id': '

In [8]:
available_fields = sorted({
    key
    for record in lab_records
    for key in record.keys()
})

for field in available_fields:
    print(field)

_version_
antibiotic
date_inserted
date_modified
evidence
genome_id
genome_name
id
laboratory_typing_method
laboratory_typing_method_version
laboratory_typing_platform
measurement
measurement_sign
measurement_unit
measurement_value
owner
pmid
public
resistant_phenotype
taxon_id
testing_standard
testing_standard_year
vendor


In [9]:
import time
import requests
import pandas as pd

API_BASE = "https://www.bv-brc.org/api"
AMR_ENDPOINT = f"{API_BASE}/genome_amr/"

selected_fields = [
    "genome_id",
    "genome_name",
    "taxon_id",
    "antibiotic",
    "resistant_phenotype",
    "measurement",
    "measurement_sign",
    "measurement_value",
    "measurement_unit",
    "laboratory_typing_method",
    "laboratory_typing_method_version",
    "laboratory_typing_platform",
    "testing_standard",
    "testing_standard_year",
    "vendor",
    "evidence",
    "pmid",
    "public",
]

In [10]:
field_string = ",".join(selected_fields)

test_query = (
    "eq(taxon_id,562)"
    "&eq(evidence,Laboratory Method)"
    "&in(resistant_phenotype,(Resistant,Susceptible))"
    f"&select({field_string})"
    "&limit(20)"
    "&http_accept=application/json"
)

response = requests.get(
    AMR_ENDPOINT + "?" + test_query,
    timeout=60,
)

response.raise_for_status()

ecoli_test = pd.DataFrame(response.json())

print(ecoli_test.shape)
ecoli_test.head()

(20, 17)


,measurement,laboratory_typing_method,taxon_id,measurement_value,evidence,resistant_phenotype,public,vendor,measurement_sign,antibiotic,testing_standard,pmid,genome_id,genome_name,laboratory_typing_platform,measurement_unit,testing_standard_year
0,,Disk diffusion,562,,Laboratory Method,Resistant,True,Oxoid,,ciprofloxacin,EUCAST,[30092762],562.56783,Escherichia coli strain 372-13,NaN,NaN,NaN
1,,Disk diffusion,562,,Laboratory Method,Resistant,True,Oxoid,,cefotaxime,EUCAST,[30092762],562.56881,Escherichia coli strain RL107,NaN,NaN,NaN
2,32,Broth dilution,562,32,Laboratory Method,Resistant,True,BioMerieux,=,ceftazidime,CLSI,NaN,562.42684,Escherichia coli strain 236,VITEK 2,mg/L,NaN
3,,Broth dilution,562,,Laboratory Method,Resistant,True,BioMerieux,,piperacillin/tazobactam,EUCAST,[36916881],562.144164,Escherichia coli ERR1981602,VITEK 2,NaN,2016.0
4,,Broth dilution,562,,Laboratory Method,Resistant,True,BioMerieux,,tetracycline,EUCAST,[30127495],562.7641,Escherichia coli strain 401686,VITEK 2,NaN,NaN


In [11]:
print(ecoli_test["taxon_id"].value_counts(dropna=False))
print(ecoli_test["evidence"].value_counts(dropna=False))
print(ecoli_test["resistant_phenotype"].value_counts(dropna=False))

taxon_id
562    20
Name: count, dtype: int64
evidence
Laboratory Method    20
Name: count, dtype: int64
resistant_phenotype
Resistant    20
Name: count, dtype: int64


In [12]:
def test_phenotype(phenotype, limit=20):
    query = (
        "eq(taxon_id,562)"
        "&eq(evidence,Laboratory Method)"
        f"&eq(resistant_phenotype,{phenotype})"
        f"&select({field_string})"
        f"&limit({limit})"
        "&http_accept=application/json"
    )

    response = requests.get(
        AMR_ENDPOINT + "?" + query,
        timeout=60,
    )
    response.raise_for_status()

    df = pd.DataFrame(response.json())

    print(f"{phenotype}: {len(df)} rows")
    print(df["resistant_phenotype"].value_counts(dropna=False))

    return df

In [13]:
resistant_test = test_phenotype("Resistant")
susceptible_test = test_phenotype("Susceptible")

Resistant: 20 rows
resistant_phenotype
Resistant    20
Name: count, dtype: int64
Susceptible: 20 rows
resistant_phenotype
Susceptible    20
Name: count, dtype: int64


In [14]:
import time
import requests
import pandas as pd

def download_one_phenotype(
    phenotype,
    endpoint,
    fields,
    page_size=10000,
    pause_seconds=0.5,
):
    pages = []
    start = 0
    field_string = ",".join(fields)

    while True:
        query = (
            "eq(taxon_id,562)"
            "&eq(evidence,Laboratory Method)"
            f"&eq(resistant_phenotype,{phenotype})"
            f"&select({field_string})"
            "&sort(+genome_id,+antibiotic)"
            f"&limit({page_size},{start})"
            "&http_accept=application/json"
        )

        response = requests.get(
            endpoint + "?" + query,
            timeout=180,
        )
        response.raise_for_status()

        records = response.json()

        print(
            f"{phenotype} | offset {start:,}: "
            f"{len(records):,} rows"
        )

        if not records:
            break

        pages.append(pd.DataFrame(records))

        if len(records) < page_size:
            break

        start += page_size
        time.sleep(pause_seconds)

    if not pages:
        return pd.DataFrame()

    return pd.concat(pages, ignore_index=True)

In [15]:
resistant_ast = download_one_phenotype(
    phenotype="Resistant",
    endpoint=AMR_ENDPOINT,
    fields=selected_fields,
)

susceptible_ast = download_one_phenotype(
    phenotype="Susceptible",
    endpoint=AMR_ENDPOINT,
    fields=selected_fields,
)

Resistant | offset 0: 10,000 rows
Resistant | offset 10,000: 10,000 rows
Resistant | offset 20,000: 1,273 rows
Susceptible | offset 0: 10,000 rows
Susceptible | offset 10,000: 10,000 rows
Susceptible | offset 20,000: 10,000 rows
Susceptible | offset 30,000: 10,000 rows
Susceptible | offset 40,000: 10,000 rows
Susceptible | offset 50,000: 10,000 rows
Susceptible | offset 60,000: 10,000 rows
Susceptible | offset 70,000: 1,179 rows


In [16]:
ecoli_ast = pd.concat(
    [resistant_ast, susceptible_ast],
    ignore_index=True,
)

print("\nFinal counts:")
print(ecoli_ast["resistant_phenotype"].value_counts())

print("\nTotal AST rows:", len(ecoli_ast))
print("Unique genomes:", ecoli_ast["genome_id"].nunique())


Final counts:
resistant_phenotype
Susceptible    71179
Resistant      21273
Name: count, dtype: int64

Total AST rows: 92452
Unique genomes: 8725


In [17]:
from pathlib import Path

RAW_DIR = Path(
    "/content/drive/MyDrive/genome-firewall/data/raw"
)
RAW_DIR.mkdir(parents=True, exist_ok=True)

ecoli_ast.to_csv(
    RAW_DIR / "bvbrc_ecoli_lab_ast_all.csv",
    index=False,
)

ecoli_ast.to_parquet(
    RAW_DIR / "bvbrc_ecoli_lab_ast_all.parquet",
    index=False,
)

print("Saved successfully.")

Saved successfully.


## Choose 5 antibiotics

In [18]:
import pandas as pd

df = pd.read_parquet(
    "/content/drive/MyDrive/genome-firewall/data/raw/bvbrc_ecoli_lab_ast_all.parquet"
)

print(df.shape)
df.head()

(92452, 17)


,public,resistant_phenotype,measurement_value,taxon_id,evidence,measurement,laboratory_typing_method,testing_standard,genome_name,genome_id,measurement_sign,antibiotic,pmid,testing_standard_year,measurement_unit,laboratory_typing_platform,vendor
0,True,Resistant,,562,Laboratory Method,,Disk diffusion,EUCAST,Escherichia coli c0fb0992-7bb9-11e9-a8d3-68b59...,562.100000,,ampicillin,None,NaN,None,None,None
1,True,Resistant,,562,Laboratory Method,,Disk diffusion,EUCAST,Escherichia coli c0fb0992-7bb9-11e9-a8d3-68b59...,562.100000,,trimethoprim/sulfamethoxazole,None,NaN,None,None,None
2,True,Resistant,,562,Laboratory Method,,Disk diffusion,EUCAST,Escherichia coli ca7bc2ea-7bb9-11e9-a8d3-68b59...,562.100001,,ampicillin,None,NaN,None,None,None
3,True,Resistant,,562,Laboratory Method,,Disk diffusion,EUCAST,Escherichia coli ca7bc2ea-7bb9-11e9-a8d3-68b59...,562.100001,,trimethoprim/sulfamethoxazole,None,NaN,None,None,None
4,True,Resistant,,562,Laboratory Method,,Disk diffusion,EUCAST,Escherichia coli ca96892c-7bb9-11e9-a8d3-68b59...,562.100002,,ampicillin,None,NaN,None,None,None


In [19]:
df.isna().sum().sort_values(ascending=False)

,0
testing_standard_year,75079
measurement_unit,72382
laboratory_typing_platform,58413
vendor,55872
pmid,36687
testing_standard,33149
laboratory_typing_method,3016
measurement,11
public,0
genome_name,0


In [20]:
antibiotic_counts = (
    df.groupby(["antibiotic", "resistant_phenotype"])
      .size()
      .unstack(fill_value=0)
)

antibiotic_counts["Total"] = antibiotic_counts.sum(axis=1)

antibiotic_counts = antibiotic_counts.sort_values(
    "Total",
    ascending=False
)

antibiotic_counts.head(30)

resistant_phenotype,Resistant,Susceptible,Total
antibiotic,,,
ciprofloxacin,2066,6559,8625
gentamicin,872,6687,7559
ceftazidime,998,6210,7208
ampicillin,3796,2906,6702
cefotaxime,1269,5317,6586
piperacillin/tazobactam,473,5850,6323
meropenem,69,5934,6003
cefuroxime,776,4314,5090
amoxicillin/clavulanic acid,1484,2940,4424


---
# Module 01 (continued) — E. coli

Everything below assumes `df` = the E. coli lab-measured AST table loaded above.

**Species: *Escherichia coli*, BV-BRC taxon_id 562.**


## B1. Pick antibiotics from the data (not hardcoded)

In [21]:
# Candidate antibiotics are selected FROM the data, not hardcoded.
# Two rules:
#   n_genomes    >= MIN_GENOMES   -> enough data to train + hold out
#   minority_frac >= MIN_MINORITY -> both R and S actually represented

MIN_GENOMES = 200
MIN_MINORITY = 0.10

counts = (
    df.groupby(["antibiotic", "resistant_phenotype"])["genome_id"]
      .nunique()
      .unstack(fill_value=0)
)

for col in ["Resistant", "Susceptible"]:
    if col not in counts.columns:
        counts[col] = 0

counts["n_genomes"] = counts["Resistant"] + counts["Susceptible"]
counts["minority_frac"] = (
    counts[["Resistant", "Susceptible"]].min(axis=1)
    / counts["n_genomes"].replace(0, pd.NA)
)

eligible = (
    counts[
        (counts["n_genomes"] >= MIN_GENOMES)
        & (counts["minority_frac"] >= MIN_MINORITY)
    ]
    .sort_values("n_genomes", ascending=False)
)

print("Eligible antibiotics:", len(eligible))
display(eligible[["Resistant", "Susceptible", "n_genomes", "minority_frac"]])

Eligible antibiotics: 31


resistant_phenotype,Resistant,Susceptible,n_genomes,minority_frac
antibiotic,,,,
ciprofloxacin,1864,5479,7343,0.253847
gentamicin,862,6478,7340,0.117439
ceftazidime,991,6032,7023,0.141108
cefotaxime,1268,5317,6585,0.192559
ampicillin,3666,2838,6504,0.436347
cefuroxime,712,3966,4678,0.152202
amoxicillin/clavulanic acid,1482,2926,4408,0.336207
trimethoprim/sulfamethoxazole,1436,2704,4140,0.346860
tobramycin,366,1223,1589,0.230334


In [22]:
# Look at the eligible table above, then confirm the final list here.
# The brief asks for 3-5 antibiotics on ONE species.

TARGET_ANTIBIOTICS = [
    "ampicillin",
    "ciprofloxacin",
    "gentamicin",
    "cefotaxime",
    "trimethoprim/sulfamethoxazole",
]

missing = [a for a in TARGET_ANTIBIOTICS if a not in eligible.index]
if missing:
    print("WARNING - not eligible, reconsider:", missing)

selected_ast = df[df["antibiotic"].isin(TARGET_ANTIBIOTICS)].copy()

# Keep only clean binary labels; Intermediate is dropped on purpose.
selected_ast = selected_ast[
    selected_ast["resistant_phenotype"].isin(["Resistant", "Susceptible"])
]

# One label per (genome, antibiotic). Drop genomes with conflicting results.
before = len(selected_ast)
selected_ast = (
    selected_ast
    .drop_duplicates(subset=["genome_id", "antibiotic", "resistant_phenotype"])
)
dupes = (
    selected_ast.groupby(["genome_id", "antibiotic"])
    .size().loc[lambda s: s > 1].index
)
selected_ast = selected_ast[
    ~selected_ast.set_index(["genome_id", "antibiotic"]).index.isin(dupes)
]

print(f"Rows: {before:,} -> {len(selected_ast):,}  (dropped {len(dupes):,} conflicting pairs)")
print("Unique genomes:", selected_ast["genome_id"].nunique())

display(
    selected_ast.groupby(["antibiotic", "resistant_phenotype"])
    .size().unstack(fill_value=0)
)

Rows: 33,614 -> 31,862  (dropped 25 conflicting pairs)
Unique genomes: 8711


resistant_phenotype,Resistant,Susceptible
antibiotic,,
ampicillin,3659,2831
cefotaxime,1268,5317
ciprofloxacin,1846,5461
gentamicin,862,6478
trimethoprim/sulfamethoxazole,1436,2704


In [23]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/genome-firewall")
PROCESSED_DIR = PROJECT / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR              # <- this was undefined before
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

genome_ids = (
    selected_ast["genome_id"].astype(str)
    .drop_duplicates().sort_values().reset_index(drop=True)
)

selected_ast.to_csv(PROCESSED_DIR / "ecoli_selected_ast.csv", index=False)
genome_ids.to_csv(PROCESSED_DIR / "ecoli_genome_ids.txt", index=False, header=False)

print("Genomes to download:", len(genome_ids))

Genomes to download: 8711


## B2. Install AMRFinderPlus

This cell was missing before — the pipeline called `conda run -n amr_env amrfinder`
but nothing ever created that environment. Takes ~5 min, run once per Colab session.

In [24]:
%%bash
set -e
cd /content
if [ ! -d /content/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
  mkdir -p /content/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/micromamba
./bin/micromamba create -y -n amr -c conda-forge -c bioconda ncbi-amrfinderplus -q
./bin/micromamba run -n amr amrfinder -u   # download the AMR database

bin/micromamba


warning  libmamba Failed to parse field 'noarch' (msgpack type=1) in shard package record for 'perl-net-http-6.19-pl526_0.tar.bz2': Expected STR or BIN type for string conversion. This field will be ignored.
warning  libmamba Failed to parse field 'noarch' (msgpack type=1) in shard package record for 'perl-net-http-6.19-pl5321hdfd78af_1.tar.bz2': Expected STR or BIN type for string conversion. This field will be ignored.
warning  libmamba Failed to parse field 'noarch' (msgpack type=1) in shard package record for 'perl-net-http-6.19-pl526_0.tar.bz2': Expected STR or BIN type for string conversion. This field will be ignored.
warning  libmamba Failed to parse field 'noarch' (msgpack type=1) in shard package record for 'perl-net-http-6.19-pl5321hdfd78af_1.tar.bz2': Expected STR or BIN type for string conversion. This field will be ignored.
Running: amrfinder -u
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software direc

In [25]:
import subprocess, os

MAMBA = "/content/bin/micromamba"
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"

def amrfinder(args):
    return subprocess.run(
        [MAMBA, "run", "-n", "amr", "amrfinder"] + args,
        capture_output=True, text=True,
    )

print(amrfinder(["--version"]).stdout)

# Confirm the exact organism string AMRFinderPlus expects for E. coli
print(amrfinder(["--list_organisms"]).stdout)

4.2.7


Available --organism options: Acinetobacter_baumannii, Bordetella_pertussis, Burkholderia_cepacia, Burkholderia_mallei, Burkholderia_pseudomallei, Campylobacter, Citrobacter_freundii, Clostridioides_difficile, Corynebacterium_diphtheriae, Enterobacter_asburiae, Enterobacter_cloacae, Enterococcus_faecalis, Enterococcus_faecium, Escherichia, Haemophilus_influenzae, Helicobacter_pylori, Klebsiella_oxytoca, Klebsiella_pneumoniae, Neisseria_gonorrhoeae, Neisseria_meningitidis, Pseudomonas_aeruginosa, Salmonella, Serratia_marcescens, Staphylococcus_aureus, Staphylococcus_epidermidis, Staphylococcus_pseudintermedius, Streptococcus_agalactiae, Streptococcus_pneumoniae, Streptococcus_pyogenes, Vibrio_cholerae, Vibrio_parahaemolyticus, Vibrio_vulnificus



In [26]:
# Set this from the --list_organisms output above.
# Recent AMRFinderPlus versions use "Escherichia".
ORGANISM = "Escherichia"

## B3. Download 2000 genome FASTAs

In [29]:
from pathlib import Path
from tqdm.auto import tqdm

PROJECT = Path("/content/drive/MyDrive/genome-firewall")
PROCESSED_DIR = PROJECT / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MAX_GENOMES = 2000
RANDOM_SEED = 42

all_genome_ids = (
    selected_ast["genome_id"]
    .astype(str)
    .drop_duplicates()
)

genome_ids = (
    all_genome_ids
    .sample(
        n=min(MAX_GENOMES, len(all_genome_ids)),
        random_state=RANDOM_SEED,
    )
    .sort_values()
    .reset_index(drop=True)
)

selected_ast = selected_ast[
    selected_ast["genome_id"].astype(str).isin(genome_ids)
].copy()

selected_ast.to_csv(
    PROCESSED_DIR / "ecoli_selected_ast.csv",
    index=False,
)

genome_ids.to_csv(
    PROCESSED_DIR / "ecoli_genome_ids.txt",
    index=False,
    header=False,
)

print("Available unique genomes:", len(all_genome_ids))
print("Genomes selected for download:", len(genome_ids))
print("Remaining AST rows:", len(selected_ast))

Available unique genomes: 8711
Genomes selected for download: 2000
Remaining AST rows: 7329


In [30]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import subprocess
import pandas as pd
from tqdm.auto import tqdm

FASTA_DIR = Path("/content/ecoli_fasta")
FASTA_DIR.mkdir(parents=True, exist_ok=True)

def fetch_one(genome_id):
    genome_id = str(genome_id)
    out = FASTA_DIR / f"{genome_id}.fna"

    if out.exists() and out.stat().st_size > 1000:
        return genome_id, True, "cached"

    url = (
        f"ftp://ftp.bv-brc.org/genomes/"
        f"{genome_id}/{genome_id}.fna"
    )

    result = subprocess.run(
        [
            "curl",
            "--ftp-ssl",
            "-L",
            "--fail",
            "-s",
            "--retry",
            "3",
            "--max-time",
            "120",
            url,
            "-o",
            str(out),
        ],
        capture_output=True,
        text=True,
    )

    success = (
        result.returncode == 0
        and out.exists()
        and out.stat().st_size > 1000
    )

    if not success and out.exists():
        out.unlink()

    note = "downloaded" if success else result.stderr[:120]

    return genome_id, success, note


ids = genome_ids.astype(str).tolist()

with ThreadPoolExecutor(max_workers=16) as executor:
    results = list(
        tqdm(
            executor.map(fetch_one, ids),
            total=len(ids),
            desc="Downloading E. coli genomes",
            unit="genome",
        )
    )

download_df = pd.DataFrame(
    results,
    columns=["genome_id", "success", "note"],
)

download_df.to_csv(
    PROCESSED_DIR / "ecoli_download_report.csv",
    index=False,
)

print()
print("Successful:", int(download_df["success"].sum()))
print("Failed:", int((~download_df["success"]).sum()))
print("Total attempted:", len(download_df))

display(
    download_df.loc[
        ~download_df["success"]
    ].head(20)
)


Successful: 1999
Failed: 1
Total attempted: 2000


,genome_id,success,note
348,562.145372,False,


In [31]:
from tqdm.auto import tqdm

successful_genome_ids = (
    download_df.loc[download_df["success"], "genome_id"]
    .astype(str)
    .tolist()
)

valid_fasta_paths = []

for gid in tqdm(
    successful_genome_ids,
    desc="Validating FASTA files",
    unit="genome",
):
    fasta_path = FASTA_DIR / f"{gid}.fna"

    if not fasta_path.exists():
        continue

    if fasta_path.stat().st_size <= 1000:
        continue

    try:
        with open(fasta_path, "r") as f:
            first_line = f.readline()

        if first_line.startswith(">"):
            valid_fasta_paths.append(fasta_path)

    except Exception:
        continue

print("Successful downloads:", len(successful_genome_ids))
print("Valid FASTA files:", len(valid_fasta_paths))

Validating FASTA files:   0%|          | 0/1999 [00:00<?, ?genome/s]

Successful downloads: 1999
Valid FASTA files: 1999


## B4. Run AMRFinderPlus on each genome


## AMRFinderPlus Feature Generation

The AMR feature extraction step was performed externally on a Linux cloud server due to the computational requirements of running AMRFinderPlus on nearly 2,000 genomes.

A custom Bash pipeline was used to:

1. Download each *E. coli* genome assembly from the BV-BRC FTP server.
2. Install and configure AMRFinderPlus using Micromamba.
3. Run AMRFinderPlus (`--plus`) on each genome in parallel (`--organism Escherichia`).
4. Package all annotation results into a compressed archive.

The resulting `amr_out.tar.gz` archive was then transferred to Google Colab and used as the input for the downstream feature engineering and machine learning pipeline.

> This preprocessing step was executed on a cloud server, while all subsequent data processing, model training, and evaluation were performed in this notebook.

### Reproducibility

The Bash script used for the external AMRFinderPlus preprocessing (`run_amr.sh`) is included in the project repository as supplementary material. It was executed on a Linux cloud server prior to running this notebook.

In [37]:
!mkdir -p /content/bin

!curl -Ls \
  https://micro.mamba.pm/api/micromamba/linux-64/latest \
  | tar -xvj -C /content/bin bin/micromamba

!chmod +x /content/bin/bin/micromamba

bin/micromamba


In [38]:
!/content/bin/bin/micromamba --version

2.8.1


In [41]:
import subprocess
import os

MAMBA = "/content/bin/micromamba"
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"

def amrfinder(args):
    return subprocess.run(
        [MAMBA, "run", "-n", "amr", "amrfinder"] + args,
        capture_output=True,
        text=True,
    )

In [45]:
!/content/bin/micromamba create \
  -y \
  -r /content/micromamba \
  -n amr \
  --channel-priority strict \
  -c conda-forge \
  -c bioconda \
  ncbi-amrfinderplus

Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.0 sec)
Using Flat Repodata for conda-forge/linux-64                                              ✔ Done (8.2 sec)
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Fetching and Parsing Packages' Shards                                                     ✔ Done (0.0 sec)
Using Flat Repodata for conda-forge/noarch                                                ✔ Done (4.9 sec)
Using Cached Shard Index for bioconda/linux-64                                                      ✔ Done
Using Cached Shard Index for bioconda

## Load TSV extracted from server

In [61]:
from pathlib import Path
import shutil
import tarfile

# ============================================================
# BLOCK 1: EXTRACT NEW 2K AMRFINDER ARCHIVE
# ============================================================

ARCHIVE_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/amr_full_2k.tar.gz"
)

EXTRACT_DIR = Path(
    "/content/drive/MyDrive/genome-firewall/amr_full_2k_extracted"
)

if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(
        f"Archive not found:\n{ARCHIVE_PATH}"
    )

# Remove old extraction so old and new TSV files cannot mix
if EXTRACT_DIR.exists():
    print("Removing previous extraction folder...")
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("EXTRACTING NEW AMRFINDER ARCHIVE")
print("=" * 70)
print("Archive:", ARCHIVE_PATH)
print("Destination:", EXTRACT_DIR)

with tarfile.open(ARCHIVE_PATH, mode="r:gz") as archive:
    try:
        archive.extractall(
            path=EXTRACT_DIR,
            filter="data",
        )
    except TypeError:
        # Compatibility with older Python versions
        archive.extractall(path=EXTRACT_DIR)

# Find real TSV files only
# Ignore macOS metadata such as ._filename.tsv and __MACOSX folders
tsv_files = sorted(
    path
    for path in EXTRACT_DIR.rglob("*.tsv")
    if not path.name.startswith("._")
    and "__MACOSX" not in path.parts
    and path.is_file()
)

print("\n" + "=" * 70)
print("EXTRACTION FINISHED")
print("=" * 70)
print("TSV files found:", len(tsv_files))

if len(tsv_files) == 0:
    raise RuntimeError(
        "No TSV files were found after extraction."
    )

print("\nFirst 10 TSV filenames:")
for path in tsv_files[:10]:
    print(" ", path.name)

print("\nLast 5 TSV filenames:")
for path in tsv_files[-5:]:
    print(" ", path.name)

if len(tsv_files) < 1900:
    print(
        "\nWARNING: Fewer than 1,900 TSV files were found. "
        "Check whether the archive is complete."
    )
elif len(tsv_files) > 2100:
    print(
        "\nWARNING: More than 2,100 TSV files were found. "
        "Check for duplicate folders or metadata files."
    )
else:
    print("\nTSV count looks appropriate for the 2,000-genome dataset.")

EXTRACTING NEW AMRFINDER ARCHIVE
Archive: /content/drive/MyDrive/genome-firewall/amr_full_2k.tar.gz
Destination: /content/drive/MyDrive/genome-firewall/amr_full_2k_extracted

EXTRACTION FINISHED
TSV files found: 1996

First 10 TSV filenames:
  562.100000.tsv
  562.100002.tsv
  562.100005.tsv
  562.100008.tsv
  562.100018.tsv
  562.100023.tsv
  562.100030.tsv
  562.100032.tsv
  562.100033.tsv
  562.100047.tsv

Last 5 TSV filenames:
  562.99971.tsv
  562.99972.tsv
  562.99981.tsv
  562.99986.tsv
  562.99992.tsv

TSV count looks appropriate for the 2,000-genome dataset.


In [62]:
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# ============================================================
# BLOCK 2: BUILD 2K AMR FEATURE MATRIX
# ============================================================

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_amr_feature_matrix_2k.csv"
)

MARKER_TABLE_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_amr_markers_long_2k.csv"
)

FEATURE_COUNTS_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_amr_feature_counts_2k.csv"
)

# Retain features found in at least this many genomes
MIN_GENOME_COUNT = 2

# Confirm that Block 1 has been run
if "tsv_files" not in globals():
    raise RuntimeError(
        "tsv_files does not exist. Run Block 1 first."
    )

if len(tsv_files) == 0:
    raise RuntimeError(
        "tsv_files is empty. Run Block 1 again."
    )

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_text(value):
    """Convert a TSV value into a clean string."""
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.lower() in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a",
    }:
        return ""

    return value


def sanitize_feature_text(value):
    """
    Make feature names safe as CSV column names while preserving
    useful gene and mutation information.
    """
    value = clean_text(value)

    value = re.sub(
        r"\s+",
        "_",
        value,
    )

    value = re.sub(
        r"[^A-Za-z0-9_.:+\-]+",
        "_",
        value,
    )

    value = re.sub(
        r"_+",
        "_",
        value,
    )

    return value.strip("_")


def find_column(columns, candidates):
    """
    Find the first matching column, case-insensitively.
    """
    normalized = {
        str(column).strip().lower(): column
        for column in columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in normalized:
            return normalized[key]

    return None


def get_genome_id_from_filename(path):
    """
    Preserve genome IDs exactly as written in the TSV filename.

    Example:
        562.100000.tsv -> 562.100000
    """
    genome_id = path.name

    if genome_id.lower().endswith(".tsv"):
        genome_id = genome_id[:-4]

    return genome_id.strip()


# ============================================================
# PROCESS ALL TSV FILES
# ============================================================

marker_rows = []
failed_files = []
successful_files = 0

for output_path in tqdm(
    tsv_files,
    desc="Processing AMRFinder TSVs",
    unit="genome",
):

    genome_id = get_genome_id_from_filename(
        output_path
    )

    try:
        result_df = pd.read_csv(
            output_path,
            sep="\t",
            dtype=str,
            keep_default_na=False,
            low_memory=False,
        )
    except Exception as error:
        failed_files.append(
            {
                "file": str(output_path),
                "error": str(error),
            }
        )
        continue

    successful_files += 1

    # Empty TSV means AMRFinder found no reportable elements.
    # The genome will still be included later as an all-zero row.
    if result_df.empty:
        continue

    element_symbol_column = find_column(
        result_df.columns,
        [
            "Element symbol",
            "element_symbol",
        ],
    )

    element_type_column = find_column(
        result_df.columns,
        [
            "Type",
            "element_type",
        ],
    )

    element_subtype_column = find_column(
        result_df.columns,
        [
            "Subtype",
            "element_subtype",
        ],
    )

    element_name_column = find_column(
        result_df.columns,
        [
            "Element name",
            "element_name",
        ],
    )

    if element_symbol_column is None:
        failed_files.append(
            {
                "file": str(output_path),
                "error": (
                    "Missing required AMRFinder column: "
                    "'Element symbol'"
                ),
            }
        )
        successful_files -= 1
        continue

    for _, row in result_df.iterrows():

        symbol = clean_text(
            row.get(element_symbol_column, "")
        )

        element_type = clean_text(
            row.get(element_type_column, "")
            if element_type_column is not None
            else ""
        )

        subtype = clean_text(
            row.get(element_subtype_column, "")
            if element_subtype_column is not None
            else ""
        )

        element_name = clean_text(
            row.get(element_name_column, "")
            if element_name_column is not None
            else ""
        )

        if not symbol:
            continue

        # --plus can report virulence and stress-response elements.
        # Keep only antimicrobial-resistance features.
        if element_type and element_type.upper() != "AMR":
            continue

        clean_symbol = sanitize_feature_text(
            symbol
        )

        clean_name = sanitize_feature_text(
            element_name
        )

        if subtype.upper() == "POINT":
            # Preserve the specific mutation description when available.
            # Example:
            # MUTATION_gyrA_S83L
            mutation_detail = clean_name

            if mutation_detail:
                feature = (
                    f"MUTATION_{clean_symbol}_"
                    f"{mutation_detail}"
                )
            else:
                feature = (
                    f"MUTATION_{clean_symbol}"
                )
        else:
            feature = (
                f"GENE_{clean_symbol}"
            )

        marker_rows.append(
            {
                "genome_id": genome_id,
                "feature": feature,
                "element_type": element_type,
                "element_subtype": subtype,
                "element_symbol": symbol,
                "element_name": element_name,
            }
        )

# ============================================================
# CREATE LONG MARKER TABLE
# ============================================================

markers = pd.DataFrame(
    marker_rows,
    columns=[
        "genome_id",
        "feature",
        "element_type",
        "element_subtype",
        "element_symbol",
        "element_name",
    ],
)

if not markers.empty:
    markers = (
        markers
        .drop_duplicates(
            subset=[
                "genome_id",
                "feature",
            ]
        )
        .reset_index(drop=True)
    )

# Include every TSV genome, even if it had no AMR marker
all_genome_ids = [
    get_genome_id_from_filename(path)
    for path in tsv_files
]

all_genome_ids = list(
    dict.fromkeys(all_genome_ids)
)

# ============================================================
# COUNT FEATURE PREVALENCE
# ============================================================

if markers.empty:
    raise RuntimeError(
        "No AMR markers were extracted from any TSV file."
    )

feature_counts = (
    markers
    .groupby("feature")["genome_id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("genome_count")
    .reset_index()
)

retained_features = feature_counts.loc[
    feature_counts["genome_count"] >= MIN_GENOME_COUNT,
    "feature",
].tolist()

removed_features = feature_counts.loc[
    feature_counts["genome_count"] < MIN_GENOME_COUNT,
    "feature",
].tolist()

if len(retained_features) == 0:
    raise RuntimeError(
        "No AMR features remained after prevalence filtering."
    )

# ============================================================
# BUILD BINARY FEATURE MATRIX
# ============================================================

retained_markers = markers[
    markers["feature"].isin(retained_features)
].copy()

retained_markers["value"] = np.uint8(1)

feature_matrix = (
    retained_markers
    .pivot_table(
        index="genome_id",
        columns="feature",
        values="value",
        aggfunc="max",
        fill_value=0,
    )
    .astype(np.uint8)
)

# Reindex to ensure genomes with zero retained markers are present
feature_matrix = feature_matrix.reindex(
    all_genome_ids,
    fill_value=0,
)

feature_matrix.index.name = "genome_id"

feature_matrix = (
    feature_matrix
    .sort_index(axis=1)
    .reset_index()
)

# Explicitly preserve genome IDs as text
feature_matrix["genome_id"] = (
    feature_matrix["genome_id"]
    .astype("string")
)

# ============================================================
# SAVE OUTPUTS
# ============================================================

feature_matrix.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False,
)

markers.to_csv(
    MARKER_TABLE_OUTPUT_PATH,
    index=False,
)

feature_counts.to_csv(
    FEATURE_COUNTS_OUTPUT_PATH,
    index=False,
)

if failed_files:
    failed_output_path = (
        OUTPUT_DIR / "module02_failed_amrfinder_tsvs_2k.csv"
    )

    pd.DataFrame(failed_files).to_csv(
        failed_output_path,
        index=False,
    )

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FEATURE MATRIX FINISHED")
print("=" * 70)

print("TSVs found:", len(tsv_files))
print("Successfully processed:", successful_files)
print("Failed:", len(failed_files))
print("Unique TSV genome IDs:", len(all_genome_ids))
print("Marker observations:", len(markers))
print("Distinct raw features:", len(feature_counts))
print(
    f"Features retained (present in >= {MIN_GENOME_COUNT} genomes):",
    len(retained_features),
)
print("Rare features removed:", len(removed_features))
print("Genomes in matrix:", feature_matrix["genome_id"].nunique())
print("Feature matrix shape:", feature_matrix.shape)

print("\nTop 20 AMR features:")
display(feature_counts.head(20))

print("\nSaved feature matrix:")
print(FEATURE_OUTPUT_PATH)

print("\nSaved long marker table:")
print(MARKER_TABLE_OUTPUT_PATH)

print("\nSaved feature counts:")
print(FEATURE_COUNTS_OUTPUT_PATH)

if failed_files:
    print("\nWARNING: Some files failed.")
    display(pd.DataFrame(failed_files).head(20))

Processing AMRFinder TSVs:   0%|          | 0/1996 [00:00<?, ?genome/s]


FEATURE MATRIX FINISHED
TSVs found: 1996
Successfully processed: 1996
Failed: 0
Unique TSV genome IDs: 1996
Marker observations: 19092
Distinct raw features: 434
Features retained (present in >= 2 genomes): 240
Rare features removed: 194
Genomes in matrix: 1996
Feature matrix shape: (1996, 241)

Top 20 AMR features:


,feature,genome_count
0,GENE_acrF,1981
1,GENE_blaEC,1665
2,GENE_emrD,1463
3,GENE_mdtM,1287
4,GENE_blaTEM-1,747
5,GENE_sul2,620
6,GENE_aph_3_-Ib,603
7,GENE_aph_6_-Id,593
8,MUTATION_gyrA_S83L_Escherichia_quinolone_resis...,583
9,MUTATION_uhpT_E350Q_Escherichia_fosfomycin_res...,557



Saved feature matrix:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_amr_feature_matrix_2k.csv

Saved long marker table:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_amr_markers_long_2k.csv

Saved feature counts:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_amr_feature_counts_2k.csv


## Feature Extractions

In [63]:
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# BLOCK 3: MERGE 2K AMR FEATURES WITH E. COLI AST LABELS
# ============================================================

FEATURE_MATRIX_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs/"
    "module02_amr_feature_matrix_2k.csv"
)

AST_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/data/processed/"
    "ecoli_selected_ast.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINING_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_training_input_2k.csv"
)

CLEAN_AST_OUTPUT_PATH = (
    OUTPUT_DIR / "ecoli_ast_clean_for_training_2k.csv"
)

LABEL_BALANCE_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_label_balance_2k.csv"
)

MATCH_REPORT_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_genome_match_report_2k.csv"
)

# ============================================================
# CHECK INPUT FILES
# ============================================================

if not FEATURE_MATRIX_PATH.exists():
    raise FileNotFoundError(
        f"Feature matrix not found:\n{FEATURE_MATRIX_PATH}"
    )

if not AST_PATH.exists():
    raise FileNotFoundError(
        f"AST file not found:\n{AST_PATH}"
    )

# ============================================================
# LOAD BOTH FILES WITH GENOME IDs AS STRINGS
# ============================================================

# Critical:
# dtype={"genome_id": "string"} prevents IDs such as
# 562.100000 from becoming float 562.1.

feature_matrix = pd.read_csv(
    FEATURE_MATRIX_PATH,
    dtype={
        "genome_id": "string",
    },
    keep_default_na=False,
    low_memory=False,
)

ast = pd.read_csv(
    AST_PATH,
    dtype={
        "genome_id": "string",
    },
    keep_default_na=False,
    low_memory=False,
)

print("=" * 70)
print("FILES LOADED")
print("=" * 70)
print("Feature matrix shape:", feature_matrix.shape)
print("AST file shape:", ast.shape)

print("\nFeature matrix first genome IDs:")
print(
    feature_matrix["genome_id"]
    .head(10)
    .tolist()
)

print("\nAST first genome IDs:")
print(
    ast["genome_id"]
    .head(10)
    .tolist()
)

# ============================================================
# NORMALIZE COLUMN NAMES
# ============================================================

ast.columns = (
    ast.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        r"[^a-z0-9]+",
        "_",
        regex=True,
    )
    .str.strip("_")
)

feature_matrix.columns = [
    str(column).strip()
    for column in feature_matrix.columns
]

print("\nAST columns:")
print(ast.columns.tolist())

# ============================================================
# FIND REQUIRED AST COLUMNS
# ============================================================

def find_first_existing_column(
    columns,
    candidates,
):
    for candidate in candidates:
        if candidate in columns:
            return candidate

    return None


genome_column = find_first_existing_column(
    ast.columns,
    [
        "genome_id",
        "genomeid",
        "genome",
    ],
)

antibiotic_column = find_first_existing_column(
    ast.columns,
    [
        "antibiotic",
        "antibiotic_name",
        "drug",
        "drug_name",
    ],
)

phenotype_column = find_first_existing_column(
    ast.columns,
    [
        "resistant_phenotype",
        "phenotype",
        "resistance_phenotype",
        "ast_phenotype",
    ],
)

if genome_column is None:
    raise KeyError(
        "Could not find the genome ID column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

if antibiotic_column is None:
    raise KeyError(
        "Could not find the antibiotic column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

if phenotype_column is None:
    raise KeyError(
        "Could not find the phenotype column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

print("\nDetected AST columns:")
print("Genome column:", genome_column)
print("Antibiotic column:", antibiotic_column)
print("Phenotype column:", phenotype_column)

ast = ast.rename(
    columns={
        genome_column: "genome_id",
        antibiotic_column: "antibiotic",
        phenotype_column: "resistant_phenotype",
    }
)

# ============================================================
# CLEAN VALUES WITHOUT CONVERTING GENOME IDS TO FLOAT
# ============================================================

feature_matrix["genome_id"] = (
    feature_matrix["genome_id"]
    .str.strip()
)

ast["genome_id"] = (
    ast["genome_id"]
    .str.strip()
)

ast["antibiotic"] = (
    ast["antibiotic"]
    .astype("string")
    .str.strip()
    .str.lower()
)

ast["resistant_phenotype"] = (
    ast["resistant_phenotype"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Remove blank IDs
feature_matrix = feature_matrix[
    feature_matrix["genome_id"].notna()
    & feature_matrix["genome_id"].ne("")
].copy()

ast = ast[
    ast["genome_id"].notna()
    & ast["genome_id"].ne("")
].copy()

# Check uniqueness of feature-matrix genome IDs
duplicate_feature_ids = (
    feature_matrix["genome_id"]
    .duplicated(keep=False)
)

if duplicate_feature_ids.any():
    duplicate_values = (
        feature_matrix.loc[
            duplicate_feature_ids,
            "genome_id",
        ]
        .value_counts()
    )

    print("\nDuplicate feature-matrix genome IDs:")
    display(duplicate_values.head(20))

    raise ValueError(
        "The feature matrix contains duplicate genome IDs."
    )

# ============================================================
# CONVERT PHENOTYPE INTO BINARY LABEL
# ============================================================

phenotype_map = {
    "susceptible": 0,
    "s": 0,

    "resistant": 1,
    "r": 1,

    "non-susceptible": 1,
    "non susceptible": 1,
    "nonsusceptible": 1,
}

ast["label"] = (
    ast["resistant_phenotype"]
    .map(phenotype_map)
)

print("\nOriginal phenotype values:")
print(
    ast["resistant_phenotype"]
    .value_counts(dropna=False)
)

unrecognized_phenotypes = (
    ast.loc[
        ast["label"].isna(),
        "resistant_phenotype",
    ]
    .value_counts(dropna=False)
)

if not unrecognized_phenotypes.empty:
    print("\nPhenotype values that will be ignored:")
    print(unrecognized_phenotypes)

ast = ast.dropna(
    subset=[
        "genome_id",
        "antibiotic",
        "label",
    ]
).copy()

ast["label"] = (
    ast["label"]
    .astype(np.uint8)
)

# ============================================================
# REMOVE CONFLICTING GENOME-ANTIBIOTIC LABELS
# ============================================================

label_counts = (
    ast
    .groupby(
        [
            "genome_id",
            "antibiotic",
        ]
    )["label"]
    .nunique()
)

conflicting_pairs = (
    label_counts[
        label_counts > 1
    ]
    .reset_index()[
        [
            "genome_id",
            "antibiotic",
        ]
    ]
)

print(
    "\nConflicting genome-antibiotic pairs:",
    len(conflicting_pairs),
)

if not conflicting_pairs.empty:
    ast = ast.merge(
        conflicting_pairs.assign(
            conflicting_label=True
        ),
        on=[
            "genome_id",
            "antibiotic",
        ],
        how="left",
    )

    ast = (
        ast.loc[
            ast["conflicting_label"].isna()
        ]
        .drop(
            columns=[
                "conflicting_label",
            ]
        )
    )

# One label row for each genome-antibiotic pair
ast_clean = (
    ast[
        [
            "genome_id",
            "antibiotic",
            "resistant_phenotype",
            "label",
        ]
    ]
    .drop_duplicates(
        subset=[
            "genome_id",
            "antibiotic",
            "label",
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# CHECK GENOME-ID OVERLAP
# ============================================================

feature_genomes = set(
    feature_matrix["genome_id"].tolist()
)

ast_genomes = set(
    ast_clean["genome_id"].tolist()
)

matched_genomes = (
    feature_genomes
    & ast_genomes
)

feature_only_genomes = (
    feature_genomes
    - ast_genomes
)

ast_only_genomes = (
    ast_genomes
    - feature_genomes
)

print("\n" + "=" * 70)
print("GENOME OVERLAP")
print("=" * 70)

print("Feature matrix genomes:", len(feature_genomes))
print("AST genomes:", len(ast_genomes))
print("Matched genomes:", len(matched_genomes))
print("Feature-only genomes:", len(feature_only_genomes))
print("AST-only genomes:", len(ast_only_genomes))

match_rate = (
    len(matched_genomes)
    / len(feature_genomes)
    if len(feature_genomes) > 0
    else 0
)

print(
    "Feature-genome match rate:",
    f"{match_rate:.2%}",
)

if len(matched_genomes) == 0:
    raise ValueError(
        "No genome IDs matched between the feature matrix "
        "and the AST file."
    )

if match_rate < 0.95:
    print(
        "\nWARNING: Fewer than 95% of feature-matrix genomes "
        "matched the AST file."
    )

# Save genome-level match report
match_report_rows = []

for genome_id in sorted(feature_genomes | ast_genomes):

    if (
        genome_id in feature_genomes
        and genome_id in ast_genomes
    ):
        match_status = "matched"
    elif genome_id in feature_genomes:
        match_status = "feature_only"
    else:
        match_status = "ast_only"

    match_report_rows.append(
        {
            "genome_id": genome_id,
            "match_status": match_status,
        }
    )

match_report = pd.DataFrame(
    match_report_rows
)

# ============================================================
# MERGE LABELS WITH FEATURES
# ============================================================

training_table = ast_clean.merge(
    feature_matrix,
    on="genome_id",
    how="inner",
    validate="many_to_one",
)

feature_columns = [
    column
    for column in feature_matrix.columns
    if column != "genome_id"
]

training_table = training_table[
    [
        "genome_id",
        "antibiotic",
        "resistant_phenotype",
        "label",
    ]
    + feature_columns
]

# Cara's code calls the binary target "y".
# Keep both "label" and "y" for compatibility.
training_table["y"] = (
    training_table["label"]
    .astype(np.uint8)
)

# ============================================================
# LABEL BALANCE PER ANTIBIOTIC
# ============================================================

label_balance = (
    training_table
    .groupby(
        [
            "antibiotic",
            "label",
        ]
    )["genome_id"]
    .nunique()
    .unstack(fill_value=0)
    .rename(
        columns={
            0: "susceptible",
            1: "resistant",
        }
    )
)

if "susceptible" not in label_balance.columns:
    label_balance["susceptible"] = 0

if "resistant" not in label_balance.columns:
    label_balance["resistant"] = 0

label_balance["total"] = (
    label_balance["susceptible"]
    + label_balance["resistant"]
)

label_balance["resistant_fraction"] = np.where(
    label_balance["total"] > 0,
    label_balance["resistant"]
    / label_balance["total"],
    np.nan,
)

label_balance = (
    label_balance
    .sort_values(
        by="total",
        ascending=False,
    )
)

# ============================================================
# SAVE FINAL OUTPUTS
# ============================================================

ast_clean.to_csv(
    CLEAN_AST_OUTPUT_PATH,
    index=False,
)

training_table.to_csv(
    TRAINING_OUTPUT_PATH,
    index=False,
)

label_balance.to_csv(
    LABEL_BALANCE_OUTPUT_PATH,
)

match_report.to_csv(
    MATCH_REPORT_OUTPUT_PATH,
    index=False,
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("MERGE FINISHED")
print("=" * 70)

print("Clean AST rows:", len(ast_clean))
print(
    "Matched genomes in training table:",
    training_table["genome_id"].nunique(),
)
print("Training rows:", len(training_table))
print(
    "Antibiotics:",
    training_table["antibiotic"].nunique(),
)
print("AMR features:", len(feature_columns))
print(
    "Training matrix shape:",
    training_table.shape,
)

print("\nLabel balance by antibiotic:")
display(label_balance)

print("\nSaved final training dataset:")
print(TRAINING_OUTPUT_PATH)

print("\nSaved clean AST:")
print(CLEAN_AST_OUTPUT_PATH)

print("\nSaved label balance:")
print(LABEL_BALANCE_OUTPUT_PATH)

print("\nSaved genome match report:")
print(MATCH_REPORT_OUTPUT_PATH)

FILES LOADED
Feature matrix shape: (1996, 241)
AST file shape: (7329, 17)

Feature matrix first genome IDs:
['562.100000', '562.100002', '562.100005', '562.100008', '562.100018', '562.100023', '562.100030', '562.100032', '562.100033', '562.100047']

AST first genome IDs:
['562.100000', '562.100000', '562.100011', '562.100023', '562.100023', '562.100023', '562.100023', '562.100023', '562.100028', '562.100030']

AST columns:
['public', 'resistant_phenotype', 'measurement_value', 'taxon_id', 'evidence', 'measurement', 'laboratory_typing_method', 'testing_standard', 'genome_name', 'genome_id', 'measurement_sign', 'antibiotic', 'pmid', 'testing_standard_year', 'measurement_unit', 'laboratory_typing_platform', 'vendor']

Detected AST columns:
Genome column: genome_id
Antibiotic column: antibiotic
Phenotype column: resistant_phenotype

Original phenotype values:
resistant_phenotype
susceptible    5230
resistant      2099
Name: count, dtype: Int64

Conflicting genome-antibiotic pairs: 0

GENOM

label,susceptible,resistant,total,resistant_fraction
antibiotic,,,,
gentamicin,326,51,377,0.135279
ciprofloxacin,274,96,370,0.259459
cefotaxime,280,73,353,0.206799
ampicillin,131,209,340,0.614706
trimethoprim/sulfamethoxazole,129,70,199,0.351759



Saved final training dataset:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_training_input_2k.csv

Saved clean AST:
/content/drive/MyDrive/genome-firewall/model_outputs/ecoli_ast_clean_for_training_2k.csv

Saved label balance:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_label_balance_2k.csv

Saved genome match report:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_genome_match_report_2k.csv


In [64]:
from pathlib import Path
import pandas as pd

# ============================================================
# DIAGNOSE WHICH AST FILE MATCHES THE NEW 2K AMRFINDER ARCHIVE
# ============================================================

FEATURE_MATRIX_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs/"
    "module02_amr_feature_matrix_2k.csv"
)

CANDIDATE_AST_FILES = [
    Path(
        "/content/drive/MyDrive/genome-firewall/data/processed/"
        "ecoli_selected_ast.csv"
    ),
    Path(
        "/content/drive/MyDrive/genome-firewall/data/raw/"
        "bvbrc_ecoli_selected5_ast.csv"
    ),
    Path(
        "/content/drive/MyDrive/genome-firewall/data/raw/"
        "bvbrc_ecoli_lab_ast_all.csv"
    ),
]

# ============================================================
# LOAD FEATURE GENOME IDS AS STRINGS
# ============================================================

feature_matrix = pd.read_csv(
    FEATURE_MATRIX_PATH,
    dtype={"genome_id": "string"},
    keep_default_na=False,
    low_memory=False,
)

feature_matrix["genome_id"] = (
    feature_matrix["genome_id"]
    .str.strip()
)

feature_genomes = set(
    feature_matrix.loc[
        feature_matrix["genome_id"].ne(""),
        "genome_id",
    ]
)

print("=" * 80)
print("AMRFINDER FEATURE MATRIX")
print("=" * 80)
print("Unique feature genomes:", len(feature_genomes))
print("First 10 IDs:", sorted(feature_genomes)[:10])

# ============================================================
# CHECK OVERLAP WITH EACH CANDIDATE AST FILE
# ============================================================

results = []

for ast_path in CANDIDATE_AST_FILES:

    if not ast_path.exists():
        results.append(
            {
                "file": str(ast_path),
                "status": "file_not_found",
                "ast_genomes": 0,
                "matched": 0,
                "feature_only": len(feature_genomes),
                "ast_only": 0,
                "feature_match_rate": 0,
            }
        )
        continue

    # Inspect header first so genome_id/genomeid can be forced to text
    header = pd.read_csv(
        ast_path,
        nrows=0,
    )

    normalized_columns = {
        str(column).strip().lower(): column
        for column in header.columns
    }

    genome_column = None

    for candidate in [
        "genome_id",
        "genomeid",
        "genome",
    ]:
        if candidate in normalized_columns:
            genome_column = normalized_columns[candidate]
            break

    if genome_column is None:
        results.append(
            {
                "file": str(ast_path),
                "status": "no_genome_column",
                "ast_genomes": 0,
                "matched": 0,
                "feature_only": len(feature_genomes),
                "ast_only": 0,
                "feature_match_rate": 0,
            }
        )
        continue

    ast_df = pd.read_csv(
        ast_path,
        dtype={genome_column: "string"},
        keep_default_na=False,
        low_memory=False,
    )

    ast_ids = (
        ast_df[genome_column]
        .str.strip()
    )

    ast_genomes = set(
        ast_ids[
            ast_ids.notna()
            & ast_ids.ne("")
        ]
    )

    matched = feature_genomes & ast_genomes
    feature_only = feature_genomes - ast_genomes
    ast_only = ast_genomes - feature_genomes

    results.append(
        {
            "file": str(ast_path),
            "status": "ok",
            "genome_column": genome_column,
            "ast_rows": len(ast_df),
            "ast_genomes": len(ast_genomes),
            "matched": len(matched),
            "feature_only": len(feature_only),
            "ast_only": len(ast_only),
            "feature_match_rate": (
                len(matched) / len(feature_genomes)
                if feature_genomes
                else 0
            ),
        }
    )

# ============================================================
# DISPLAY RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "matched",
    ascending=False,
)

print("\n" + "=" * 80)
print("OVERLAP RESULTS")
print("=" * 80)

display(results_df)

best_result = results_df.iloc[0]

print("\nBest matching file:")
print(best_result["file"])

print("Matched genomes:", best_result["matched"])
print(
    "Feature match rate:",
    f"{best_result['feature_match_rate']:.2%}",
)

# ============================================================
# SHOW EXAMPLES FROM CURRENT PROCESSED AST FILE
# ============================================================

CURRENT_AST_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/data/processed/"
    "ecoli_selected_ast.csv"
)

current_ast = pd.read_csv(
    CURRENT_AST_PATH,
    dtype={"genome_id": "string"},
    keep_default_na=False,
    low_memory=False,
)

current_ast_genomes = set(
    current_ast["genome_id"]
    .str.strip()
)

matched_current = feature_genomes & current_ast_genomes
feature_only_current = feature_genomes - current_ast_genomes
ast_only_current = current_ast_genomes - feature_genomes

print("\n" + "=" * 80)
print("EXAMPLES FOR ecoli_selected_ast.csv")
print("=" * 80)

print("\nMatched examples:")
print(sorted(matched_current)[:20])

print("\nAMRFinder-only examples:")
print(sorted(feature_only_current)[:20])

print("\nAST-only examples:")
print(sorted(ast_only_current)[:20])

AMRFINDER FEATURE MATRIX
Unique feature genomes: 1996
First 10 IDs: ['562.100000', '562.100002', '562.100005', '562.100008', '562.100018', '562.100023', '562.100030', '562.100032', '562.100033', '562.100047']

OVERLAP RESULTS


,file,status,genome_column,ast_rows,ast_genomes,matched,feature_only,ast_only,feature_match_rate
2,/content/drive/MyDrive/genome-firewall/data/ra...,ok,genome_id,92452,8725,1996,0,6729,1.000000
1,/content/drive/MyDrive/genome-firewall/data/ra...,ok,genome_id,24060,8720,1995,1,6725,0.999499
0,/content/drive/MyDrive/genome-firewall/data/pr...,ok,genome_id,7329,2000,456,1540,1544,0.228457



Best matching file:
/content/drive/MyDrive/genome-firewall/data/raw/bvbrc_ecoli_lab_ast_all.csv
Matched genomes: 1996
Feature match rate: 100.00%

EXAMPLES FOR ecoli_selected_ast.csv

Matched examples:
['562.100000', '562.100023', '562.100030', '562.100098', '562.100116', '562.100136', '562.100137', '562.100139', '562.100145', '562.100155', '562.100175', '562.100218', '562.100224', '562.100271', '562.100340', '562.100357', '562.100359', '562.100361', '562.100377', '562.100438']

AMRFinder-only examples:
['562.100002', '562.100005', '562.100008', '562.100018', '562.100032', '562.100033', '562.100047', '562.100052', '562.100064', '562.100068', '562.100072', '562.100074', '562.100075', '562.100078', '562.100079', '562.100083', '562.100099', '562.100107', '562.100110', '562.100118']

AST-only examples:
['562.100003', '562.100011', '562.100028', '562.100034', '562.100040', '562.100048', '562.100050', '562.100059', '562.100063', '562.100066', '562.100071', '562.100076', '562.100080', '562.1

In [65]:
from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# MERGE FULL 2K AMR FEATURE MATRIX WITH MATCHING RAW AST DATASET
# ============================================================

FEATURE_MATRIX_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs/"
    "module02_amr_feature_matrix_2k.csv"
)

AST_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/data/raw/"
    "bvbrc_ecoli_lab_ast_all.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINING_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_training_input_2k.csv"
)

CLEAN_AST_OUTPUT_PATH = (
    OUTPUT_DIR / "ecoli_ast_clean_for_training_2k.csv"
)

LABEL_BALANCE_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_label_balance_2k.csv"
)

MATCH_REPORT_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_genome_match_report_2k.csv"
)

# Five antibiotics used by the project
TARGET_ANTIBIOTICS = {
    "ampicillin",
    "cefotaxime",
    "ciprofloxacin",
    "gentamicin",
    "trimethoprim/sulfamethoxazole",
}

# ============================================================
# CHECK FILES
# ============================================================

if not FEATURE_MATRIX_PATH.exists():
    raise FileNotFoundError(
        f"Feature matrix not found:\n{FEATURE_MATRIX_PATH}"
    )

if not AST_PATH.exists():
    raise FileNotFoundError(
        f"AST file not found:\n{AST_PATH}"
    )

# ============================================================
# LOAD FILES WITH GENOME IDS PRESERVED AS STRINGS
# ============================================================

feature_matrix = pd.read_csv(
    FEATURE_MATRIX_PATH,
    dtype={"genome_id": "string"},
    keep_default_na=False,
    low_memory=False,
)

ast = pd.read_csv(
    AST_PATH,
    dtype={"genome_id": "string"},
    keep_default_na=False,
    low_memory=False,
)

print("=" * 75)
print("FILES LOADED")
print("=" * 75)

print("Feature matrix shape:", feature_matrix.shape)
print("AST file shape:", ast.shape)

print("\nFeature matrix first genome IDs:")
print(feature_matrix["genome_id"].head(10).tolist())

print("\nAST first genome IDs:")
print(ast["genome_id"].head(10).tolist())

# ============================================================
# NORMALIZE COLUMN NAMES
# ============================================================

ast.columns = (
    ast.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        r"[^a-z0-9]+",
        "_",
        regex=True,
    )
    .str.strip("_")
)

feature_matrix.columns = [
    str(column).strip()
    for column in feature_matrix.columns
]

print("\nAST columns:")
print(ast.columns.tolist())

# ============================================================
# FIND REQUIRED COLUMNS
# ============================================================

def find_first_existing_column(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


genome_column = find_first_existing_column(
    ast.columns,
    [
        "genome_id",
        "genomeid",
        "genome",
    ],
)

antibiotic_column = find_first_existing_column(
    ast.columns,
    [
        "antibiotic",
        "antibiotic_name",
        "drug",
        "drug_name",
    ],
)

phenotype_column = find_first_existing_column(
    ast.columns,
    [
        "resistant_phenotype",
        "phenotype",
        "resistance_phenotype",
        "ast_phenotype",
    ],
)

if genome_column is None:
    raise KeyError(
        "Could not find a genome ID column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

if antibiotic_column is None:
    raise KeyError(
        "Could not find an antibiotic column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

if phenotype_column is None:
    raise KeyError(
        "Could not find a phenotype column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

print("\nDetected columns:")
print("Genome column:", genome_column)
print("Antibiotic column:", antibiotic_column)
print("Phenotype column:", phenotype_column)

ast = ast.rename(
    columns={
        genome_column: "genome_id",
        antibiotic_column: "antibiotic",
        phenotype_column: "resistant_phenotype",
    }
)

# ============================================================
# CLEAN GENOME IDS, ANTIBIOTICS, AND PHENOTYPES
# ============================================================

feature_matrix["genome_id"] = (
    feature_matrix["genome_id"]
    .str.strip()
)

ast["genome_id"] = (
    ast["genome_id"]
    .str.strip()
)

ast["antibiotic"] = (
    ast["antibiotic"]
    .astype("string")
    .str.strip()
    .str.lower()
)

ast["resistant_phenotype"] = (
    ast["resistant_phenotype"]
    .astype("string")
    .str.strip()
    .str.lower()
)

feature_matrix = feature_matrix[
    feature_matrix["genome_id"].notna()
    & feature_matrix["genome_id"].ne("")
].copy()

ast = ast[
    ast["genome_id"].notna()
    & ast["genome_id"].ne("")
].copy()

# ============================================================
# FILTER TO THE FIVE TARGET ANTIBIOTICS
# ============================================================

print("\nRows before antibiotic filtering:", len(ast))

print("\nTop antibiotic values before filtering:")
print(
    ast["antibiotic"]
    .value_counts()
    .head(30)
)

ast = ast[
    ast["antibiotic"].isin(TARGET_ANTIBIOTICS)
].copy()

print("\nRows after filtering to five antibiotics:", len(ast))
print("Antibiotics retained:")
print(
    sorted(
        ast["antibiotic"]
        .dropna()
        .unique()
        .tolist()
    )
)

missing_antibiotics = (
    TARGET_ANTIBIOTICS
    - set(ast["antibiotic"].unique())
)

if missing_antibiotics:
    print(
        "\nWARNING: These target antibiotics were not found:",
        sorted(missing_antibiotics),
    )

# ============================================================
# CHECK FEATURE MATRIX IDS ARE UNIQUE
# ============================================================

duplicate_feature_ids = (
    feature_matrix["genome_id"]
    .duplicated(keep=False)
)

if duplicate_feature_ids.any():
    duplicate_values = (
        feature_matrix.loc[
            duplicate_feature_ids,
            "genome_id",
        ]
        .value_counts()
    )

    print("\nDuplicate feature-matrix genome IDs:")
    print(duplicate_values.head(20))

    raise ValueError(
        "The feature matrix contains duplicate genome IDs."
    )

# ============================================================
# MAP PHENOTYPES TO BINARY LABELS
# ============================================================

phenotype_map = {
    "susceptible": 0,
    "s": 0,

    "resistant": 1,
    "r": 1,

    "non-susceptible": 1,
    "non susceptible": 1,
    "nonsusceptible": 1,
}

ast["label"] = (
    ast["resistant_phenotype"]
    .map(phenotype_map)
)

print("\nPhenotype values:")
print(
    ast["resistant_phenotype"]
    .value_counts(dropna=False)
)

unrecognized_phenotypes = (
    ast.loc[
        ast["label"].isna(),
        "resistant_phenotype",
    ]
    .value_counts(dropna=False)
)

if not unrecognized_phenotypes.empty:
    print("\nUnrecognized phenotype values that will be removed:")
    print(unrecognized_phenotypes)

ast = ast.dropna(
    subset=[
        "genome_id",
        "antibiotic",
        "label",
    ]
).copy()

ast["label"] = (
    ast["label"]
    .astype(np.uint8)
)

# ============================================================
# REMOVE CONFLICTING GENOME-ANTIBIOTIC LABELS
# ============================================================

label_counts = (
    ast
    .groupby(
        [
            "genome_id",
            "antibiotic",
        ]
    )["label"]
    .nunique()
)

conflicting_pairs = (
    label_counts[
        label_counts > 1
    ]
    .reset_index()[
        [
            "genome_id",
            "antibiotic",
        ]
    ]
)

print(
    "\nConflicting genome-antibiotic pairs:",
    len(conflicting_pairs),
)

if not conflicting_pairs.empty:
    ast = ast.merge(
        conflicting_pairs.assign(
            conflicting_label=True
        ),
        on=[
            "genome_id",
            "antibiotic",
        ],
        how="left",
    )

    ast = (
        ast.loc[
            ast["conflicting_label"].isna()
        ]
        .drop(
            columns=["conflicting_label"]
        )
    )

# Keep one clean row per genome-antibiotic-label combination
ast_clean = (
    ast[
        [
            "genome_id",
            "antibiotic",
            "resistant_phenotype",
            "label",
        ]
    ]
    .drop_duplicates(
        subset=[
            "genome_id",
            "antibiotic",
            "label",
        ]
    )
    .reset_index(drop=True)
)

# ============================================================
# CHECK GENOME OVERLAP
# ============================================================

feature_genomes = set(
    feature_matrix["genome_id"].tolist()
)

ast_genomes = set(
    ast_clean["genome_id"].tolist()
)

matched_genomes = (
    feature_genomes
    & ast_genomes
)

feature_only_genomes = (
    feature_genomes
    - ast_genomes
)

ast_only_genomes = (
    ast_genomes
    - feature_genomes
)

print("\n" + "=" * 75)
print("GENOME OVERLAP")
print("=" * 75)

print("Feature matrix genomes:", len(feature_genomes))
print("AST genomes after filtering:", len(ast_genomes))
print("Matched genomes:", len(matched_genomes))
print("Feature-only genomes:", len(feature_only_genomes))
print("AST-only genomes:", len(ast_only_genomes))

match_rate = (
    len(matched_genomes) / len(feature_genomes)
    if feature_genomes
    else 0
)

print(
    "Feature-genome match rate:",
    f"{match_rate:.2%}",
)

if len(matched_genomes) == 0:
    raise ValueError(
        "No genomes matched between the feature matrix and AST data."
    )

# ============================================================
# BUILD MATCH REPORT
# ============================================================

match_report_rows = []

for genome_id in sorted(
    feature_genomes | ast_genomes
):
    if (
        genome_id in feature_genomes
        and genome_id in ast_genomes
    ):
        status = "matched"
    elif genome_id in feature_genomes:
        status = "feature_only"
    else:
        status = "ast_only"

    match_report_rows.append(
        {
            "genome_id": genome_id,
            "match_status": status,
        }
    )

match_report = pd.DataFrame(
    match_report_rows
)

# ============================================================
# MERGE LABELS WITH AMR FEATURES
# ============================================================

training_table = ast_clean.merge(
    feature_matrix,
    on="genome_id",
    how="inner",
    validate="many_to_one",
)

feature_columns = [
    column
    for column in feature_matrix.columns
    if column != "genome_id"
]

training_table = training_table[
    [
        "genome_id",
        "antibiotic",
        "resistant_phenotype",
        "label",
    ]
    + feature_columns
]

training_table["y"] = (
    training_table["label"]
    .astype(np.uint8)
)

# ============================================================
# LABEL BALANCE
# ============================================================

label_balance = (
    training_table
    .groupby(
        [
            "antibiotic",
            "label",
        ]
    )["genome_id"]
    .nunique()
    .unstack(fill_value=0)
    .rename(
        columns={
            0: "susceptible",
            1: "resistant",
        }
    )
)

if "susceptible" not in label_balance.columns:
    label_balance["susceptible"] = 0

if "resistant" not in label_balance.columns:
    label_balance["resistant"] = 0

label_balance["total"] = (
    label_balance["susceptible"]
    + label_balance["resistant"]
)

label_balance["resistant_fraction"] = np.where(
    label_balance["total"] > 0,
    label_balance["resistant"]
    / label_balance["total"],
    np.nan,
)

label_balance = (
    label_balance
    .sort_values(
        by="total",
        ascending=False,
    )
)

# ============================================================
# SAVE OUTPUTS
# ============================================================

ast_clean.to_csv(
    CLEAN_AST_OUTPUT_PATH,
    index=False,
)

training_table.to_csv(
    TRAINING_OUTPUT_PATH,
    index=False,
)

label_balance.to_csv(
    LABEL_BALANCE_OUTPUT_PATH,
)

match_report.to_csv(
    MATCH_REPORT_OUTPUT_PATH,
    index=False,
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("MERGE FINISHED")
print("=" * 75)

print("Clean AST rows:", len(ast_clean))
print(
    "Matched genomes in training table:",
    training_table["genome_id"].nunique(),
)
print("Training rows:", len(training_table))
print(
    "Antibiotics:",
    training_table["antibiotic"].nunique(),
)
print("AMR features:", len(feature_columns))
print(
    "Training matrix shape:",
    training_table.shape,
)

print("\nLabel balance by antibiotic:")
display(label_balance)

print("\nSaved final training dataset:")
print(TRAINING_OUTPUT_PATH)

print("\nSaved clean AST:")
print(CLEAN_AST_OUTPUT_PATH)

print("\nSaved label balance:")
print(LABEL_BALANCE_OUTPUT_PATH)

print("\nSaved genome match report:")
print(MATCH_REPORT_OUTPUT_PATH)

FILES LOADED
Feature matrix shape: (1996, 241)
AST file shape: (92452, 17)

Feature matrix first genome IDs:
['562.100000', '562.100002', '562.100005', '562.100008', '562.100018', '562.100023', '562.100030', '562.100032', '562.100033', '562.100047']

AST first genome IDs:
['562.100000', '562.100000', '562.100001', '562.100001', '562.100002', '562.100002', '562.100005', '562.100006', '562.100007', '562.100007']

AST columns:
['public', 'resistant_phenotype', 'measurement_value', 'taxon_id', 'evidence', 'measurement', 'laboratory_typing_method', 'testing_standard', 'genome_name', 'genome_id', 'measurement_sign', 'antibiotic', 'pmid', 'testing_standard_year', 'measurement_unit', 'laboratory_typing_platform', 'vendor']

Detected columns:
Genome column: genome_id
Antibiotic column: antibiotic
Phenotype column: resistant_phenotype

Rows before antibiotic filtering: 92452

Top antibiotic values before filtering:
antibiotic
ciprofloxacin                    8625
gentamicin                      

label,susceptible,resistant,total,resistant_fraction
antibiotic,,,,
ciprofloxacin,1261,407,1668,0.244005
gentamicin,1473,190,1663,0.114251
cefotaxime,1226,303,1529,0.198169
ampicillin,651,846,1497,0.565130
trimethoprim/sulfamethoxazole,619,326,945,0.344974



Saved final training dataset:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_training_input_2k.csv

Saved clean AST:
/content/drive/MyDrive/genome-firewall/model_outputs/ecoli_ast_clean_for_training_2k.csv

Saved label balance:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_label_balance_2k.csv

Saved genome match report:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_genome_match_report_2k.csv


In [59]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# STAGE 3: MERGE AMR FEATURE MATRIX WITH E. COLI AST LABELS
# ============================================================

FEATURE_MATRIX_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs/"
    "module02_amr_feature_matrix.csv"
)

AST_PATH = Path(
    "/content/drive/MyDrive/genome-firewall/data/processed/"
    "ecoli_selected_ast.csv"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/genome-firewall/model_outputs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_OUTPUT_PATH = (
    OUTPUT_DIR / "module02_training_input.csv"
)

CLEAN_AST_OUTPUT_PATH = (
    OUTPUT_DIR / "ecoli_ast_clean_for_training.csv"
)

# ============================================================
# LOAD FILES
# ============================================================

if not FEATURE_MATRIX_PATH.exists():
    raise FileNotFoundError(
        f"Feature matrix not found:\n{FEATURE_MATRIX_PATH}"
    )

if not AST_PATH.exists():
    raise FileNotFoundError(
        f"AST file not found:\n{AST_PATH}"
    )

feature_matrix = pd.read_csv(
    FEATURE_MATRIX_PATH,
    dtype={"genome_id": str},
)

ast = pd.read_csv(
    AST_PATH,
    dtype={"genome_id": str},
)

print("=" * 70)
print("FILES LOADED")
print("=" * 70)
print("Feature matrix shape:", feature_matrix.shape)
print("AST file shape:", ast.shape)

# ============================================================
# NORMALIZE COLUMN NAMES
# ============================================================

ast.columns = (
    ast.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

feature_matrix.columns = [
    str(column).strip()
    for column in feature_matrix.columns
]

print("\nAST columns:")
print(ast.columns.tolist())

# ============================================================
# DETECT IMPORTANT AST COLUMNS
# ============================================================

def find_first_existing_column(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


genome_column = find_first_existing_column(
    ast.columns,
    [
        "genome_id",
        "genomeid",
        "genome",
    ],
)

antibiotic_column = find_first_existing_column(
    ast.columns,
    [
        "antibiotic",
        "antibiotic_name",
        "drug",
        "drug_name",
    ],
)

phenotype_column = find_first_existing_column(
    ast.columns,
    [
        "resistant_phenotype",
        "phenotype",
        "resistance_phenotype",
        "ast_phenotype",
    ],
)

if genome_column is None:
    raise KeyError(
        "Could not find the genome ID column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

if antibiotic_column is None:
    raise KeyError(
        "Could not find the antibiotic column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

if phenotype_column is None:
    raise KeyError(
        "Could not find the phenotype column.\n"
        f"Available columns: {ast.columns.tolist()}"
    )

print("\nDetected AST columns:")
print("Genome column:", genome_column)
print("Antibiotic column:", antibiotic_column)
print("Phenotype column:", phenotype_column)

ast = ast.rename(
    columns={
        genome_column: "genome_id",
        antibiotic_column: "antibiotic",
        phenotype_column: "phenotype",
    }
)

# ============================================================
# CLEAN GENOME IDS, ANTIBIOTICS, AND PHENOTYPES
# ============================================================

feature_matrix["genome_id"] = (
    feature_matrix["genome_id"]
    .astype(str)
    .str.strip()
)

ast["genome_id"] = (
    ast["genome_id"]
    .astype(str)
    .str.strip()
)

ast["antibiotic"] = (
    ast["antibiotic"]
    .astype(str)
    .str.strip()
    .str.lower()
)

ast["phenotype"] = (
    ast["phenotype"]
    .astype(str)
    .str.strip()
    .str.lower()
)

phenotype_map = {
    "s": 0,
    "susceptible": 0,

    "r": 1,
    "resistant": 1,
    "non-susceptible": 1,
    "non susceptible": 1,
    "nonsusceptible": 1,
}

ast["label"] = ast["phenotype"].map(
    phenotype_map
)

print("\nOriginal phenotype values:")
print(ast["phenotype"].value_counts(dropna=False))

unrecognized_phenotypes = (
    ast.loc[
        ast["label"].isna(),
        "phenotype",
    ]
    .value_counts(dropna=False)
)

if not unrecognized_phenotypes.empty:
    print("\nIgnored phenotype values:")
    print(unrecognized_phenotypes)

ast = ast.dropna(
    subset=[
        "genome_id",
        "antibiotic",
        "label",
    ]
).copy()

ast["label"] = ast["label"].astype(
    np.uint8
)

# ============================================================
# REMOVE CONFLICTING GENOME-ANTIBIOTIC LABELS
# ============================================================

label_counts = (
    ast
    .groupby(
        [
            "genome_id",
            "antibiotic",
        ]
    )["label"]
    .nunique()
)

conflicting_pairs = (
    label_counts[
        label_counts > 1
    ]
    .reset_index()[
        [
            "genome_id",
            "antibiotic",
        ]
    ]
)

print(
    "\nConflicting genome-antibiotic pairs:",
    len(conflicting_pairs),
)

if not conflicting_pairs.empty:
    ast = ast.merge(
        conflicting_pairs.assign(
            conflicting_label=1
        ),
        on=[
            "genome_id",
            "antibiotic",
        ],
        how="left",
    )

    ast = (
        ast[
            ast["conflicting_label"].isna()
        ]
        .drop(
            columns="conflicting_label"
        )
    )

# Keep one row per genome-antibiotic-label combination
ast_clean = (
    ast[
        [
            "genome_id",
            "antibiotic",
            "label",
        ]
    ]
    .drop_duplicates()
    .copy()
)

# ============================================================
# CHECK OVERLAP BEFORE MERGING
# ============================================================

feature_genomes = set(
    feature_matrix["genome_id"]
)

ast_genomes = set(
    ast_clean["genome_id"]
)

matched_genomes = (
    feature_genomes
    & ast_genomes
)

feature_only_genomes = (
    feature_genomes
    - ast_genomes
)

ast_only_genomes = (
    ast_genomes
    - feature_genomes
)

print("\nGenome overlap:")
print("Feature matrix genomes:", len(feature_genomes))
print("AST genomes:", len(ast_genomes))
print("Matched genomes:", len(matched_genomes))
print("Feature-only genomes:", len(feature_only_genomes))
print("AST-only genomes:", len(ast_only_genomes))

if len(matched_genomes) == 0:
    raise ValueError(
        "No genome IDs matched between the feature matrix and AST file."
    )

# ============================================================
# MERGE FEATURES WITH AST LABELS
# ============================================================

training_input = ast_clean.merge(
    feature_matrix,
    on="genome_id",
    how="inner",
    validate="many_to_one",
)

feature_columns = [
    column
    for column in feature_matrix.columns
    if column != "genome_id"
]

training_input = training_input[
    [
        "genome_id",
        "antibiotic",
        "label",
    ]
    + feature_columns
]

# ============================================================
# CREATE LABEL-BALANCE SUMMARY
# ============================================================

label_balance = (
    training_input
    .groupby(
        [
            "antibiotic",
            "label",
        ]
    )["genome_id"]
    .nunique()
    .unstack(fill_value=0)
    .rename(
        columns={
            0: "susceptible",
            1: "resistant",
        }
    )
)

if "susceptible" not in label_balance.columns:
    label_balance["susceptible"] = 0

if "resistant" not in label_balance.columns:
    label_balance["resistant"] = 0

label_balance["total"] = (
    label_balance["susceptible"]
    + label_balance["resistant"]
)

label_balance["resistant_fraction"] = np.where(
    label_balance["total"] > 0,
    label_balance["resistant"]
    / label_balance["total"],
    np.nan,
)

label_balance = label_balance.sort_values(
    "total",
    ascending=False,
)

# ============================================================
# SAVE OUTPUTS
# ============================================================

ast_clean.to_csv(
    CLEAN_AST_OUTPUT_PATH,
    index=False,
)

training_input.to_csv(
    TRAINING_OUTPUT_PATH,
    index=False,
)

label_balance.to_csv(
    OUTPUT_DIR / "module02_label_balance.csv"
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("MERGE FINISHED")
print("=" * 70)

print("Clean AST rows:", len(ast_clean))
print("Matched genomes:", training_input["genome_id"].nunique())
print("Training rows:", len(training_input))
print("Antibiotics:", training_input["antibiotic"].nunique())
print("AMR features:", len(feature_columns))
print("Training matrix shape:", training_input.shape)

print("\nLabel balance by antibiotic:")
display(label_balance)

print("\nSaved training dataset:")
print(TRAINING_OUTPUT_PATH)

print("\nSaved clean AST:")
print(CLEAN_AST_OUTPUT_PATH)

FILES LOADED
Feature matrix shape: (847, 296)
AST file shape: (7329, 17)

AST columns:
['public', 'resistant_phenotype', 'measurement_value', 'taxon_id', 'evidence', 'measurement', 'laboratory_typing_method', 'testing_standard', 'genome_name', 'genome_id', 'measurement_sign', 'antibiotic', 'pmid', 'testing_standard_year', 'measurement_unit', 'laboratory_typing_platform', 'vendor']

Detected AST columns:
Genome column: genome_id
Antibiotic column: antibiotic
Phenotype column: resistant_phenotype

Original phenotype values:
phenotype
susceptible    5230
resistant      2099
Name: count, dtype: int64

Conflicting genome-antibiotic pairs: 0

Genome overlap:
Feature matrix genomes: 847
AST genomes: 2000
Matched genomes: 191
Feature-only genomes: 656
AST-only genomes: 1809

MERGE FINISHED
Clean AST rows: 7329
Matched genomes: 191
Training rows: 674
Antibiotics: 5
AMR features: 295
Training matrix shape: (674, 298)

Label balance by antibiotic:


label,susceptible,resistant,total,resistant_fraction
antibiotic,,,,
gentamicin,136,22,158,0.139241
ciprofloxacin,110,42,152,0.276316
cefotaxime,112,35,147,0.238095
ampicillin,51,90,141,0.638298
trimethoprim/sulfamethoxazole,49,27,76,0.355263



Saved training dataset:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_training_input.csv

Saved clean AST:
/content/drive/MyDrive/genome-firewall/model_outputs/ecoli_ast_clean_for_training.csv


## Split Data

In [68]:
# ============================================================
# MODULE 03A
# CREATE SHARED GENOME-LEVEL TRAIN / VALIDATION / TEST SPLIT
#
# Output:
# genome_id, split
#
# Split:
#   70% Train
#   15% Validation
#   15% Test
#
# Every genome appears in exactly ONE split.
# ============================================================

from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/genome-firewall")

INPUT_FILE = (
    PROJECT_ROOT /
    "model_outputs" /
    "module02_training_input_2k.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "model_outputs" /
    "module03_split"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "shared_genome_split.csv"
)

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

df = pd.read_csv(
    INPUT_FILE,
    dtype={"genome_id": str},
    low_memory=False,
)

print("="*70)
print("Training data")
print("="*70)

print(df.shape)
print("Unique genomes:", df["genome_id"].nunique())

# ------------------------------------------------------------
# Unique genomes
# ------------------------------------------------------------

genomes = pd.DataFrame({
    "genome_id":
    sorted(df["genome_id"].unique())
})

# ------------------------------------------------------------
# 70% train
# ------------------------------------------------------------

gss1 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=42,
)

train_idx, temp_idx = next(
    gss1.split(
        genomes,
        groups=genomes["genome_id"]
    )
)

train = genomes.iloc[train_idx]
temp = genomes.iloc[temp_idx]

# ------------------------------------------------------------
# Remaining 30%
# -> 15% validation
# -> 15% test
# ------------------------------------------------------------

gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=43,
)

val_idx, test_idx = next(
    gss2.split(
        temp,
        groups=temp["genome_id"]
    )
)

validation = temp.iloc[val_idx]
test = temp.iloc[test_idx]

train["split"] = "train"
validation["split"] = "validation"
test["split"] = "test"

split_df = pd.concat(
    [
        train,
        validation,
        test,
    ],
    ignore_index=True,
)

split_df = split_df.sort_values(
    "genome_id"
).reset_index(drop=True)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert split_df["genome_id"].nunique() == len(split_df)
assert len(split_df) == df["genome_id"].nunique()

print("\nSplit summary")
print(split_df["split"].value_counts())

print("\nPercentages")
print(
    split_df["split"]
    .value_counts(normalize=True)
    .round(3)
)

# ------------------------------------------------------------
# Label balance
# ------------------------------------------------------------

merged = df.merge(
    split_df,
    on="genome_id",
)

label_col = "y" if "y" in merged.columns else "label"

balance = (
    merged
    .groupby(
        ["antibiotic","split",label_col]
    )
    .size()
    .unstack(fill_value=0)
)

print("\nLabel balance")
print(balance)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

split_df.to_csv(
    OUTPUT_FILE,
    index=False,
)

print("\nSaved to:")
print(OUTPUT_FILE)

print("\nPreview")
display(split_df.head())

Training data
(7302, 245)
Unique genomes: 1996

Split summary
split
train         1397
test           300
validation     299
Name: count, dtype: int64

Percentages
split
train         0.70
test          0.15
validation    0.15
Name: proportion, dtype: float64

Label balance
y                                            0    1
antibiotic                    split                
ampicillin                    test          89  127
                              train        470  587
                              validation    92  132
cefotaxime                    test         183   48
                              train        852  207
                              validation   191   48
ciprofloxacin                 test         194   67
                              train        873  274
                              validation   194   66
gentamicin                    test         230   28
                              train       1013  131
                              validation   230   

,genome_id,split
0,562.100000,train
1,562.100002,train
2,562.100005,test
3,562.100008,train
4,562.100018,train


## TRAIN MODELS AND GENERATE HELD-OUT TEST PREDICTIONS

In [69]:
# ============================================================
# MODULE 04
# TRAIN MODELS AND GENERATE HELD-OUT TEST PREDICTIONS
#
# Candidate models:
#   1. Logistic Regression
#   2. Random Forest
#   3. Extra Trees
#
# Model selection:
#   Best validation balanced accuracy at threshold 0.50
#
# Confidence / no-call rule:
#   y_prob >= 0.80 -> confident Resistant
#   y_prob <= 0.20 -> confident Susceptible
#   otherwise      -> no-call
#
# Important:
#   Test data is never used for model selection.
#
# Main output for Cara's evaluator:
#   data/predictions/test_predictions.csv
#
# Required prediction columns:
#   genome_id
#   antibiotic
#   y_true
#   y_prob
#
# Optional confidence column:
#   is_called
# ============================================================

from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/genome-firewall"
)

TRAINING_DATA_PATH = (
    PROJECT_ROOT
    / "model_outputs"
    / "module02_training_input_2k.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "model_outputs"
    / "module03_split"
    / "shared_genome_split.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "model_outputs"
    / "module04_training"
)

MODEL_DIR = (
    OUTPUT_DIR
    / "trained_models"
)

PREDICTION_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTION_PATH = (
    PREDICTION_DIR
    / "test_predictions.csv"
)

VALIDATION_RESULTS_PATH = (
    OUTPUT_DIR
    / "validation_model_comparison.csv"
)

MODEL_SELECTION_PATH = (
    OUTPUT_DIR
    / "selected_models.csv"
)

TRAINING_SUMMARY_PATH = (
    OUTPUT_DIR
    / "training_summary.json"
)

# Standard binary prediction threshold
PREDICTION_THRESHOLD = 0.50

# Confidence/no-call thresholds
CONFIDENT_SUSCEPTIBLE_THRESHOLD = 0.20
CONFIDENT_RESISTANT_THRESHOLD = 0.80

RANDOM_STATE = 42

# ============================================================
# CHECK INPUT FILES
# ============================================================

if not TRAINING_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Training data not found:\n"
        f"{TRAINING_DATA_PATH}"
    )

if not SPLIT_PATH.exists():
    raise FileNotFoundError(
        f"Shared split not found:\n"
        f"{SPLIT_PATH}"
    )

# ============================================================
# LOAD DATA
# ============================================================

training_df = pd.read_csv(
    TRAINING_DATA_PATH,
    dtype={
        "genome_id": str,
        "antibiotic": str,
    },
    low_memory=False,
)

split_df = pd.read_csv(
    SPLIT_PATH,
    dtype={
        "genome_id": str,
        "split": str,
    },
)

training_df["genome_id"] = (
    training_df["genome_id"]
    .astype(str)
    .str.strip()
)

training_df["antibiotic"] = (
    training_df["antibiotic"]
    .astype(str)
    .str.strip()
    .str.lower()
)

split_df["genome_id"] = (
    split_df["genome_id"]
    .astype(str)
    .str.strip()
)

split_df["split"] = (
    split_df["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("=" * 90)
print("MODULE 04: MODEL TRAINING")
print("=" * 90)

print("\nTraining data:")
print(TRAINING_DATA_PATH)

print("\nShared split:")
print(SPLIT_PATH)

print("\nTraining shape:")
print(training_df.shape)

print("\nSplit shape:")
print(split_df.shape)

# ============================================================
# IDENTIFY TARGET COLUMN
# ============================================================

if "y" in training_df.columns:
    TARGET_COLUMN = "y"

elif "label" in training_df.columns:
    TARGET_COLUMN = "label"

else:
    raise KeyError(
        "Could not find target column 'y' or 'label'."
    )

training_df[TARGET_COLUMN] = pd.to_numeric(
    training_df[TARGET_COLUMN],
    errors="coerce",
)

training_df = training_df.dropna(
    subset=[
        "genome_id",
        "antibiotic",
        TARGET_COLUMN,
    ]
).copy()

training_df[TARGET_COLUMN] = (
    training_df[TARGET_COLUMN]
    .astype(int)
)

invalid_labels = (
    set(training_df[TARGET_COLUMN].unique())
    - {0, 1}
)

if invalid_labels:
    raise ValueError(
        f"Invalid target values: {sorted(invalid_labels)}"
    )

# ============================================================
# VALIDATE SPLIT FILE
# ============================================================

valid_splits = {
    "train",
    "validation",
    "test",
}

invalid_splits = (
    set(split_df["split"].unique())
    - valid_splits
)

if invalid_splits:
    raise ValueError(
        f"Invalid split names: {sorted(invalid_splits)}"
    )

if split_df["genome_id"].duplicated().any():
    duplicated_genomes = (
        split_df.loc[
            split_df["genome_id"].duplicated(),
            "genome_id",
        ]
        .tolist()
    )

    raise ValueError(
        "A genome occurs more than once in the split file. "
        f"Examples: {duplicated_genomes[:10]}"
    )

# ============================================================
# MERGE SHARED SPLIT
# ============================================================

data = training_df.merge(
    split_df,
    on="genome_id",
    how="left",
    validate="many_to_one",
)

missing_split_count = (
    data["split"]
    .isna()
    .sum()
)

if missing_split_count > 0:
    missing_genomes = (
        data.loc[
            data["split"].isna(),
            "genome_id",
        ]
        .drop_duplicates()
        .tolist()
    )

    raise ValueError(
        f"{missing_split_count} rows have no split assignment.\n"
        f"Example genomes: {missing_genomes[:10]}"
    )

print("\nMerged data shape:")
print(data.shape)

print("\nUnique genomes:")
print(data["genome_id"].nunique())

print("\nRows per split:")
print(data["split"].value_counts())

print("\nGenomes per split:")
print(
    data.groupby("split")["genome_id"]
    .nunique()
)

# ============================================================
# VERIFY NO GENOME LEAKAGE
# ============================================================

train_genomes = set(
    data.loc[
        data["split"] == "train",
        "genome_id",
    ]
)

validation_genomes = set(
    data.loc[
        data["split"] == "validation",
        "genome_id",
    ]
)

test_genomes = set(
    data.loc[
        data["split"] == "test",
        "genome_id",
    ]
)

assert train_genomes.isdisjoint(validation_genomes)
assert train_genomes.isdisjoint(test_genomes)
assert validation_genomes.isdisjoint(test_genomes)

print("\nGenome leakage check passed.")

print(
    "Train / validation overlap:",
    len(train_genomes & validation_genomes),
)

print(
    "Train / test overlap:",
    len(train_genomes & test_genomes),
)

print(
    "Validation / test overlap:",
    len(validation_genomes & test_genomes),
)

# ============================================================
# IDENTIFY FEATURE COLUMNS
# ============================================================

metadata_columns = {
    "genome_id",
    "antibiotic",
    "resistant_phenotype",
    "phenotype",
    "label",
    "y",
    "split",
    "group_id",
}

feature_columns = [
    column
    for column in data.columns
    if column not in metadata_columns
]

if len(feature_columns) == 0:
    raise ValueError(
        "No AMR feature columns were found."
    )

for column in feature_columns:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce",
    ).fillna(0)

data[feature_columns] = (
    data[feature_columns]
    .clip(lower=0, upper=1)
    .astype(np.uint8)
)

print("\nTarget column:")
print(TARGET_COLUMN)

print("\nNumber of AMR features:")
print(len(feature_columns))

print("\nFirst 20 feature columns:")
print(feature_columns[:20])

print("\nAntibiotics:")
print(
    sorted(
        data["antibiotic"]
        .unique()
        .tolist()
    )
)

# ============================================================
# DEFINE CANDIDATE MODELS
# ============================================================

candidate_models = {

    "logistic_regression": Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(
                    with_mean=False
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=1.0,
                    penalty="l2",
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),

    "extra_trees": ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

print("\nCandidate models:")

for model_name in candidate_models:
    print(" ", model_name)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_resistant_probability(
    model,
    X,
):
    """
    Return probability of class 1 = Resistant.
    """

    probabilities = model.predict_proba(X)

    model_classes = list(model.classes_)

    if 1 not in model_classes:
        return np.zeros(
            len(X),
            dtype=float,
        )

    resistant_index = model_classes.index(1)

    return probabilities[
        :,
        resistant_index,
    ]


def create_confidence_columns(
    y_prob,
):
    """
    Create standard predictions and confidence/no-call outputs.

    Confident Resistant:
        y_prob >= 0.80

    Confident Susceptible:
        y_prob <= 0.20

    No-call:
        0.20 < y_prob < 0.80
    """

    y_prob = np.asarray(
        y_prob,
        dtype=float,
    )

    predicted_label = (
        y_prob >= PREDICTION_THRESHOLD
    ).astype(int)

    is_called = (
        (y_prob <= CONFIDENT_SUSCEPTIBLE_THRESHOLD)
        |
        (y_prob >= CONFIDENT_RESISTANT_THRESHOLD)
    )

    confident_prediction = np.where(
        y_prob >= CONFIDENT_RESISTANT_THRESHOLD,
        1,
        np.where(
            y_prob <= CONFIDENT_SUSCEPTIBLE_THRESHOLD,
            0,
            np.nan,
        ),
    )

    prediction_text = np.where(
        y_prob >= CONFIDENT_RESISTANT_THRESHOLD,
        "Resistant",
        np.where(
            y_prob <= CONFIDENT_SUSCEPTIBLE_THRESHOLD,
            "Susceptible",
            "No-call",
        ),
    )

    confidence = np.maximum(
        y_prob,
        1 - y_prob,
    )

    confidence_margin = np.abs(
        y_prob - 0.5
    )

    return {
        "predicted_label":
            predicted_label,

        "is_called":
            is_called,

        "confident_prediction":
            confident_prediction,

        "prediction_text":
            prediction_text,

        "confidence":
            confidence,

        "confidence_margin":
            confidence_margin,
    }

# ============================================================
# TRAIN ONE MODEL PER ANTIBIOTIC
# ============================================================

validation_results = []
selected_model_rows = []
test_prediction_frames = []

selected_models = {}

antibiotics = sorted(
    data["antibiotic"]
    .dropna()
    .unique()
    .tolist()
)

print("\n" + "=" * 90)
print("TRAINING ONE MODEL PER ANTIBIOTIC")
print("=" * 90)

for antibiotic in antibiotics:

    print("\n" + "-" * 90)
    print(
        "ANTIBIOTIC:",
        antibiotic.upper(),
    )
    print("-" * 90)

    antibiotic_df = data[
        data["antibiotic"] == antibiotic
    ].copy()

    train_df = antibiotic_df[
        antibiotic_df["split"] == "train"
    ].copy()

    validation_df = antibiotic_df[
        antibiotic_df["split"] == "validation"
    ].copy()

    test_df = antibiotic_df[
        antibiotic_df["split"] == "test"
    ].copy()

    X_train = (
        train_df[feature_columns]
        .to_numpy(dtype=np.float32)
    )

    y_train = (
        train_df[TARGET_COLUMN]
        .to_numpy(dtype=int)
    )

    X_validation = (
        validation_df[feature_columns]
        .to_numpy(dtype=np.float32)
    )

    y_validation = (
        validation_df[TARGET_COLUMN]
        .to_numpy(dtype=int)
    )

    X_test = (
        test_df[feature_columns]
        .to_numpy(dtype=np.float32)
    )

    y_test = (
        test_df[TARGET_COLUMN]
        .to_numpy(dtype=int)
    )

    print("Train rows:", len(train_df))
    print("Validation rows:", len(validation_df))
    print("Test rows:", len(test_df))

    print(
        "Train S / R:",
        int((y_train == 0).sum()),
        "/",
        int((y_train == 1).sum()),
    )

    print(
        "Validation S / R:",
        int((y_validation == 0).sum()),
        "/",
        int((y_validation == 1).sum()),
    )

    print(
        "Test S / R:",
        int((y_test == 0).sum()),
        "/",
        int((y_test == 1).sum()),
    )

    if len(np.unique(y_train)) < 2:
        print(
            "Skipped: training set has only one class."
        )
        continue

    if len(validation_df) == 0:
        print(
            "Skipped: validation set is empty."
        )
        continue

    if len(test_df) == 0:
        print(
            "Skipped: test set is empty."
        )
        continue

    fitted_models = {}
    current_validation_results = []

    # --------------------------------------------------------
    # TRAIN EACH CANDIDATE MODEL
    # --------------------------------------------------------

    for model_name, model_template in candidate_models.items():

        model = clone(
            model_template
        )

        model.fit(
            X_train,
            y_train,
        )

        validation_prob = get_resistant_probability(
            model,
            X_validation,
        )

        validation_pred = (
            validation_prob
            >= PREDICTION_THRESHOLD
        ).astype(int)

        validation_balanced_accuracy = (
            balanced_accuracy_score(
                y_validation,
                validation_pred,
            )
        )

        validation_confidence = (
            create_confidence_columns(
                validation_prob
            )
        )

        validation_coverage = (
            validation_confidence[
                "is_called"
            ].mean()
        )

        validation_row = {
            "antibiotic":
                antibiotic,

            "model":
                model_name,

            "validation_balanced_accuracy":
                validation_balanced_accuracy,

            "validation_confident_coverage":
                validation_coverage,

            "prediction_threshold":
                PREDICTION_THRESHOLD,

            "confident_susceptible_threshold":
                CONFIDENT_SUSCEPTIBLE_THRESHOLD,

            "confident_resistant_threshold":
                CONFIDENT_RESISTANT_THRESHOLD,

            "n_validation_samples":
                len(validation_df),
        }

        validation_results.append(
            validation_row
        )

        current_validation_results.append(
            validation_row
        )

        fitted_models[
            model_name
        ] = model

        print(
            f"{model_name:22s}",
            "| Balanced accuracy:",
            f"{validation_balanced_accuracy:.4f}",
            "| Confident coverage:",
            f"{validation_coverage:.4f}",
        )

    # --------------------------------------------------------
    # SELECT BEST MODEL USING VALIDATION ONLY
    # --------------------------------------------------------

    current_validation_df = pd.DataFrame(
        current_validation_results
    )

    current_validation_df = (
        current_validation_df
        .sort_values(
            by=[
                "validation_balanced_accuracy",
                "validation_confident_coverage",
                "model",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    best_model_name = (
        current_validation_df.loc[
            0,
            "model",
        ]
    )

    best_validation_score = (
        current_validation_df.loc[
            0,
            "validation_balanced_accuracy",
        ]
    )

    best_validation_coverage = (
        current_validation_df.loc[
            0,
            "validation_confident_coverage",
        ]
    )

    selected_model = (
        fitted_models[
            best_model_name
        ]
    )

    selected_models[
        antibiotic
    ] = best_model_name

    selected_model_rows.append(
        {
            "antibiotic":
                antibiotic,

            "selected_model":
                best_model_name,

            "validation_balanced_accuracy":
                best_validation_score,

            "validation_confident_coverage":
                best_validation_coverage,

            "prediction_threshold":
                PREDICTION_THRESHOLD,

            "confident_susceptible_threshold":
                CONFIDENT_SUSCEPTIBLE_THRESHOLD,

            "confident_resistant_threshold":
                CONFIDENT_RESISTANT_THRESHOLD,
        }
    )

    print("\nSelected model:")
    print(best_model_name)

    print(
        "Best validation balanced accuracy:",
        round(
            best_validation_score,
            4,
        ),
    )

    print(
        "Validation confident coverage:",
        round(
            best_validation_coverage,
            4,
        ),
    )

    # --------------------------------------------------------
    # GENERATE UNTOUCHED TEST PROBABILITIES
    # --------------------------------------------------------

    test_prob = get_resistant_probability(
        selected_model,
        X_test,
    )

    confidence_outputs = (
        create_confidence_columns(
            test_prob
        )
    )

    prediction_df = pd.DataFrame(
        {
            "genome_id":
                test_df["genome_id"]
                .to_numpy(),

            "antibiotic":
                antibiotic,

            "y_true":
                y_test,

            "y_prob":
                test_prob,

            "predicted_label":
                confidence_outputs[
                    "predicted_label"
                ],

            "is_called":
                confidence_outputs[
                    "is_called"
                ],

            "confident_prediction":
                confidence_outputs[
                    "confident_prediction"
                ],

            "prediction_text":
                confidence_outputs[
                    "prediction_text"
                ],

            "confidence":
                confidence_outputs[
                    "confidence"
                ],

            "confidence_margin":
                confidence_outputs[
                    "confidence_margin"
                ],

            "model":
                best_model_name,

            "split":
                "test",
        }
    )

    test_prediction_frames.append(
        prediction_df
    )

    print(
        "\nTest confident calls:",
        int(
            prediction_df[
                "is_called"
            ].sum()
        ),
        "/",
        len(prediction_df),
    )

    print(
        "Test confident coverage:",
        round(
            prediction_df[
                "is_called"
            ].mean(),
            4,
        ),
    )

    # --------------------------------------------------------
    # REFIT SELECTED MODEL ON TRAIN + VALIDATION
    #
    # This saved model is for future predictions.
    #
    # Test probabilities above came from the model trained only
    # on the training split. Therefore the test evaluation
    # remains untouched.
    # --------------------------------------------------------

    train_validation_df = antibiotic_df[
        antibiotic_df["split"].isin(
            [
                "train",
                "validation",
            ]
        )
    ].copy()

    X_train_validation = (
        train_validation_df[
            feature_columns
        ]
        .to_numpy(dtype=np.float32)
    )

    y_train_validation = (
        train_validation_df[
            TARGET_COLUMN
        ]
        .to_numpy(dtype=int)
    )

    final_model = clone(
        candidate_models[
            best_model_name
        ]
    )

    final_model.fit(
        X_train_validation,
        y_train_validation,
    )

    safe_antibiotic_name = (
        antibiotic
        .replace("/", "_")
        .replace(" ", "_")
    )

    model_path = (
        MODEL_DIR
        / (
            f"{safe_antibiotic_name}"
            f"__{best_model_name}.joblib"
        )
    )

    joblib.dump(
        {
            "model":
                final_model,

            "antibiotic":
                antibiotic,

            "model_name":
                best_model_name,

            "feature_columns":
                feature_columns,

            "target_column":
                TARGET_COLUMN,

            "prediction_threshold":
                PREDICTION_THRESHOLD,

            "confident_susceptible_threshold":
                CONFIDENT_SUSCEPTIBLE_THRESHOLD,

            "confident_resistant_threshold":
                CONFIDENT_RESISTANT_THRESHOLD,

            "random_state":
                RANDOM_STATE,
        },
        model_path,
    )

    print("\nSaved final model:")
    print(model_path)

# ============================================================
# SAVE OUTPUTS
# ============================================================

validation_results_df = pd.DataFrame(
    validation_results
)

selected_models_df = pd.DataFrame(
    selected_model_rows
)

if len(test_prediction_frames) == 0:
    raise RuntimeError(
        "No test predictions were generated."
    )

test_predictions_df = pd.concat(
    test_prediction_frames,
    ignore_index=True,
)

# Put Cara's required columns first
prediction_first_columns = [
    "genome_id",
    "antibiotic",
    "y_true",
    "y_prob",
    "is_called",
]

prediction_remaining_columns = [
    column
    for column in test_predictions_df.columns
    if column not in prediction_first_columns
]

test_predictions_df = test_predictions_df[
    prediction_first_columns
    + prediction_remaining_columns
]

validation_results_df.to_csv(
    VALIDATION_RESULTS_PATH,
    index=False,
)

selected_models_df.to_csv(
    MODEL_SELECTION_PATH,
    index=False,
)

test_predictions_df.to_csv(
    PREDICTION_PATH,
    index=False,
)

training_summary = {
    "training_data_path":
        str(TRAINING_DATA_PATH),

    "split_path":
        str(SPLIT_PATH),

    "prediction_path":
        str(PREDICTION_PATH),

    "n_rows":
        int(len(data)),

    "n_unique_genomes":
        int(
            data["genome_id"]
            .nunique()
        ),

    "n_features":
        int(len(feature_columns)),

    "target_column":
        TARGET_COLUMN,

    "prediction_threshold":
        PREDICTION_THRESHOLD,

    "confident_susceptible_threshold":
        CONFIDENT_SUSCEPTIBLE_THRESHOLD,

    "confident_resistant_threshold":
        CONFIDENT_RESISTANT_THRESHOLD,

    "candidate_models":
        list(
            candidate_models.keys()
        ),

    "selected_models":
        selected_models,

    "random_state":
        RANDOM_STATE,
}

with open(
    TRAINING_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        training_summary,
        file,
        indent=2,
    )

# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 90)
print("TRAINING COMPLETE")
print("=" * 90)

print("\nSelected model per antibiotic:")

display(
    selected_models_df
    .sort_values("antibiotic")
    .reset_index(drop=True)
)

print("\nTest prediction preview:")

display(
    test_predictions_df.head()
)

print("\nPrediction rows:")
print(len(test_predictions_df))

print("\nTest rows per antibiotic:")

print(
    test_predictions_df[
        "antibiotic"
    ]
    .value_counts()
)

print("\nConfident coverage per antibiotic:")

coverage_summary = (
    test_predictions_df
    .groupby("antibiotic")
    .agg(
        n_test=(
            "genome_id",
            "size",
        ),
        n_called=(
            "is_called",
            "sum",
        ),
        coverage=(
            "is_called",
            "mean",
        ),
        mean_confidence=(
            "confidence",
            "mean",
        ),
    )
    .reset_index()
)

coverage_summary[
    "no_call_rate"
] = (
    1
    - coverage_summary[
        "coverage"
    ]
)

display(
    coverage_summary.round(4)
)

print("\nSaved test predictions:")
print(PREDICTION_PATH)

print("\nSaved selected models:")
print(MODEL_SELECTION_PATH)

print("\nSaved validation comparison:")
print(VALIDATION_RESULTS_PATH)

print("\nSaved trained models:")
print(MODEL_DIR)

print("\nSaved training summary:")
print(TRAINING_SUMMARY_PATH)

MODULE 04: MODEL TRAINING

Training data:
/content/drive/MyDrive/genome-firewall/model_outputs/module02_training_input_2k.csv

Shared split:
/content/drive/MyDrive/genome-firewall/model_outputs/module03_split/shared_genome_split.csv

Training shape:
(7302, 245)

Split shape:
(1996, 2)

Merged data shape:
(7302, 246)

Unique genomes:
1996

Rows per split:
split
train         5070
validation    1128
test          1104
Name: count, dtype: int64

Genomes per split:
split
test           300
train         1397
validation     299
Name: genome_id, dtype: int64

Genome leakage check passed.
Train / validation overlap: 0
Train / test overlap: 0
Validation / test overlap: 0

Target column:
y

Number of AMR features:
240

First 20 feature columns:
['GENE_aac_3_-IId', 'GENE_aac_3_-IIe', 'GENE_aac_3_-IVa', 'GENE_aac_3_-VIa', 'GENE_aac_6_-Ib', 'GENE_aac_6_-Ib-cr5', 'GENE_aac_6_-Ib3', 'GENE_aac_6_-Ib4', 'GENE_aadA1', 'GENE_aadA2', 'GENE_aadA22', 'GENE_aadA25', 'GENE_aadA4', 'GENE_aadA5', 'GENE_aadA7',

,antibiotic,selected_model,validation_balanced_accuracy,validation_confident_coverage,prediction_threshold,confident_susceptible_threshold,confident_resistant_threshold
0,ampicillin,logistic_regression,0.949605,0.964286,0.5,0.2,0.8
1,cefotaxime,logistic_regression,0.895779,0.979079,0.5,0.2,0.8
2,ciprofloxacin,random_forest,0.964542,0.903846,0.5,0.2,0.8
3,gentamicin,extra_trees,0.938569,0.758621,0.5,0.2,0.8
4,trimethoprim/sulfamethoxazole,logistic_regression,0.922986,0.902778,0.5,0.2,0.8



Test prediction preview:


,genome_id,antibiotic,y_true,y_prob,is_called,predicted_label,confident_prediction,prediction_text,confidence,confidence_margin,model,split
0,562.100005,ampicillin,1,0.079164,True,0,0.0,Susceptible,0.920836,0.420836,logistic_regression,test
1,562.100116,ampicillin,1,0.999993,True,1,1.0,Resistant,0.999993,0.499993,logistic_regression,test
2,562.100135,ampicillin,1,0.986806,True,1,1.0,Resistant,0.986806,0.486806,logistic_regression,test
3,562.100136,ampicillin,1,0.977417,True,1,1.0,Resistant,0.977417,0.477417,logistic_regression,test
4,562.100170,ampicillin,1,0.992314,True,1,1.0,Resistant,0.992314,0.492314,logistic_regression,test



Prediction rows:
1104

Test rows per antibiotic:
antibiotic
ciprofloxacin                    261
gentamicin                       258
cefotaxime                       231
ampicillin                       216
trimethoprim/sulfamethoxazole    138
Name: count, dtype: int64

Confident coverage per antibiotic:


,antibiotic,n_test,n_called,coverage,mean_confidence,no_call_rate
0,ampicillin,216,207,0.9583,0.9544,0.0417
1,cefotaxime,231,224,0.9697,0.9764,0.0303
2,ciprofloxacin,261,241,0.9234,0.9243,0.0766
3,gentamicin,258,197,0.7636,0.8635,0.2364
4,trimethoprim/sulfamethoxazole,138,126,0.9130,0.9592,0.0870



Saved test predictions:
/content/drive/MyDrive/genome-firewall/data/predictions/test_predictions.csv

Saved selected models:
/content/drive/MyDrive/genome-firewall/model_outputs/module04_training/selected_models.csv

Saved validation comparison:
/content/drive/MyDrive/genome-firewall/model_outputs/module04_training/validation_model_comparison.csv

Saved trained models:
/content/drive/MyDrive/genome-firewall/model_outputs/module04_training/trained_models

Saved training summary:
/content/drive/MyDrive/genome-firewall/model_outputs/module04_training/training_summary.json


##  EVALUATE TEST PREDICTIONS

In [70]:
# ============================================================
# MODULE 05
# EVALUATE TEST PREDICTIONS
#
# Input:
#   data/predictions/test_predictions.csv
#
# Required columns:
#   genome_id
#   antibiotic
#   y_true
#   y_prob
#
# Optional:
#   is_called
#
# Since is_called is present, the evaluator produces:
#
#   all_samples
#       Metrics on every test sample at threshold 0.50
#
#   called_only
#       Metrics only on high-confidence predictions
#
# Output:
#   results/evaluation_by_antibiotic.csv
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# ============================================================
# CARA'S EVALUATION FUNCTIONS
# ============================================================

def safe_auc(
    metric_function,
    y_true,
    y_prob,
):
    """
    AUROC and PR-AUC require both classes to be present.
    Return NaN when only one class is available.
    """

    y_true = np.asarray(
        y_true
    )

    if len(
        np.unique(y_true)
    ) < 2:

        return np.nan

    return metric_function(
        y_true,
        y_prob,
    )


def calculate_metrics(
    y_true,
    y_prob,
    threshold=0.5,
):
    """
    Calculate binary classification metrics.

    Labels
    ------
    Resistant = 1
    Susceptible = 0
    """

    y_true = np.asarray(
        y_true
    ).astype(int)

    y_prob = np.asarray(
        y_prob
    ).astype(float)

    if len(y_true) != len(y_prob):
        raise ValueError(
            "y_true and y_prob must have the same length."
        )

    if len(y_true) == 0:
        raise ValueError(
            "Cannot calculate metrics on an empty dataset."
        )

    if not np.isin(
        y_true,
        [0, 1],
    ).all():

        raise ValueError(
            "y_true must contain only 0 and 1."
        )

    if np.isnan(y_prob).any():
        raise ValueError(
            "y_prob contains missing values."
        )

    if (
        (y_prob < 0)
        |
        (y_prob > 1)
    ).any():

        raise ValueError(
            "y_prob must contain probabilities between 0 and 1."
        )

    # Convert probabilities into standard binary predictions
    y_pred = (
        y_prob >= threshold
    ).astype(int)

    # Confusion matrix:
    #
    #                 Predicted S    Predicted R
    # Actual S            TN             FP
    # Actual R            FN             TP

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    recall_r = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

    recall_s = recall_score(
        y_true,
        y_pred,
        pos_label=0,
        zero_division=0,
    )

    return {
        "n_samples":
            int(len(y_true)),

        "n_resistant":
            int(
                (y_true == 1).sum()
            ),

        "n_susceptible":
            int(
                (y_true == 0).sum()
            ),

        "true_resistant":
            int(tp),

        "true_susceptible":
            int(tn),

        "false_resistant":
            int(fp),

        "false_susceptible":
            int(fn),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "recall_R":
            recall_r,

        "recall_S":
            recall_s,

        "f1_R":
            f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0,
            ),

        "f1_S":
            f1_score(
                y_true,
                y_pred,
                pos_label=0,
                zero_division=0,
            ),

        "f1_macro":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),

        "auroc":
            safe_auc(
                roc_auc_score,
                y_true,
                y_prob,
            ),

        "pr_auc":
            safe_auc(
                average_precision_score,
                y_true,
                y_prob,
            ),

        "threshold":
            float(threshold),
    }


def evaluate_by_antibiotic(
    predictions,
    thresholds=None,
):
    """
    Evaluate predictions separately for each antibiotic.

    Required columns:
        antibiotic
        y_true
        y_prob

    Optional column:
        is_called

    If is_called is provided:
        all_samples evaluates all predictions.
        called_only evaluates high-confidence predictions only.
    """

    required_columns = {
        "antibiotic",
        "y_true",
        "y_prob",
    }

    missing_columns = (
        required_columns
        - set(predictions.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    thresholds = thresholds or {}

    results = []

    for antibiotic, group in predictions.groupby(
        "antibiotic"
    ):

        group = group.copy()

        # Remove rows without usable labels or probabilities
        group = group.dropna(
            subset=[
                "y_true",
                "y_prob",
            ]
        )

        if len(group) == 0:
            continue

        threshold = thresholds.get(
            antibiotic,
            0.5,
        )

        # ----------------------------------------------------
        # 1. EVALUATE ALL TEST SAMPLES
        # ----------------------------------------------------

        all_metrics = calculate_metrics(
            y_true=
                group["y_true"],

            y_prob=
                group["y_prob"],

            threshold=
                threshold,
        )

        all_metrics.update(
            {
                "antibiotic":
                    antibiotic,

                "evaluation_scope":
                    "all_samples",

                "coverage":
                    1.0,

                "no_call_rate":
                    0.0,
            }
        )

        results.append(
            all_metrics
        )

        # ----------------------------------------------------
        # 2. EVALUATE HIGH-CONFIDENCE CALLED SAMPLES ONLY
        # ----------------------------------------------------

        if "is_called" in group.columns:

            called_mask = (
                group["is_called"]
                .fillna(False)
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(
                    [
                        "true",
                        "1",
                        "yes",
                    ]
                )
            )

            called_group = group[
                called_mask
            ].copy()

            coverage = (
                len(called_group)
                / len(group)
                if len(group) > 0
                else np.nan
            )

            if len(called_group) > 0:

                called_metrics = calculate_metrics(
                    y_true=
                        called_group[
                            "y_true"
                        ],

                    y_prob=
                        called_group[
                            "y_prob"
                        ],

                    threshold=
                        threshold,
                )

                called_metrics.update(
                    {
                        "antibiotic":
                            antibiotic,

                        "evaluation_scope":
                            "called_only",

                        "coverage":
                            coverage,

                        "no_call_rate":
                            1 - coverage,
                    }
                )

                results.append(
                    called_metrics
                )

    results_df = pd.DataFrame(
        results
    )

    if results_df.empty:
        raise ValueError(
            "No valid prediction rows were available for evaluation."
        )

    first_columns = [
        "antibiotic",
        "evaluation_scope",
        "n_samples",
        "n_resistant",
        "n_susceptible",
        "coverage",
        "no_call_rate",
        "balanced_accuracy",
        "recall_R",
        "recall_S",
        "f1_R",
        "f1_S",
        "f1_macro",
        "auroc",
        "pr_auc",
        "threshold",
    ]

    remaining_columns = [
        column
        for column in results_df.columns
        if column not in first_columns
    ]

    return (
        results_df[
            first_columns
            + remaining_columns
        ]
        .sort_values(
            [
                "antibiotic",
                "evaluation_scope",
            ]
        )
        .reset_index(drop=True)
    )

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/genome-firewall"
)

PREDICTION_PATH = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "test_predictions.csv"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVALUATION_PATH = (
    RESULTS_DIR
    / "evaluation_by_antibiotic.csv"
)

ALL_SAMPLE_RESULTS_PATH = (
    RESULTS_DIR
    / "evaluation_all_samples.csv"
)

CALLED_ONLY_RESULTS_PATH = (
    RESULTS_DIR
    / "evaluation_called_only.csv"
)

# ============================================================
# LOAD TEST PREDICTIONS
# ============================================================

if not PREDICTION_PATH.exists():
    raise FileNotFoundError(
        f"Prediction file not found:\n"
        f"{PREDICTION_PATH}\n\n"
        "Run the training block first."
    )

predictions = pd.read_csv(
    PREDICTION_PATH,
    dtype={
        "genome_id": str,
        "antibiotic": str,
    },
)

predictions["genome_id"] = (
    predictions["genome_id"]
    .astype(str)
    .str.strip()
)

predictions["antibiotic"] = (
    predictions["antibiotic"]
    .astype(str)
    .str.strip()
    .str.lower()
)

predictions["y_true"] = pd.to_numeric(
    predictions["y_true"],
    errors="coerce",
)

predictions["y_prob"] = pd.to_numeric(
    predictions["y_prob"],
    errors="coerce",
)

print("=" * 90)
print("MODULE 05: MODEL EVALUATION")
print("=" * 90)

print("\nPrediction file:")
print(PREDICTION_PATH)

print("\nPrediction rows:")
print(len(predictions))

print("\nPrediction columns:")
print(predictions.columns.tolist())

print("\nAntibiotics:")
print(
    sorted(
        predictions["antibiotic"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nPrediction preview:")

display(
    predictions.head()
)

# ============================================================
# VALIDATE CONFIDENCE COLUMN
# ============================================================

if "is_called" not in predictions.columns:
    print(
        "\nWarning: is_called is missing. "
        "Only all-sample evaluation will be produced."
    )

else:
    called_boolean = (
        predictions["is_called"]
        .fillna(False)
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
            ]
        )
    )

    print("\nOverall confident calls:")
    print(
        int(called_boolean.sum()),
        "/",
        len(predictions),
    )

    print("\nOverall coverage:")
    print(
        round(
            called_boolean.mean(),
            4,
        )
    )

    print("\nOverall no-call rate:")
    print(
        round(
            1
            - called_boolean.mean(),
            4,
        )
    )

# ============================================================
# RUN CARA'S EVALUATION
# ============================================================

evaluation_results = evaluate_by_antibiotic(
    predictions
)

# ============================================================
# SEPARATE RESULTS
# ============================================================

all_sample_results = (
    evaluation_results[
        evaluation_results[
            "evaluation_scope"
        ] == "all_samples"
    ]
    .copy()
    .reset_index(drop=True)
)

called_only_results = (
    evaluation_results[
        evaluation_results[
            "evaluation_scope"
        ] == "called_only"
    ]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# SAVE RESULTS
# ============================================================

evaluation_results.to_csv(
    EVALUATION_PATH,
    index=False,
)

all_sample_results.to_csv(
    ALL_SAMPLE_RESULTS_PATH,
    index=False,
)

called_only_results.to_csv(
    CALLED_ONLY_RESULTS_PATH,
    index=False,
)

# ============================================================
# DISPLAY FULL RESULTS
# ============================================================

print("\n" + "=" * 90)
print("FULL HELD-OUT TEST RESULTS")
print("=" * 90)

display(
    evaluation_results.round(4)
)

# ============================================================
# DISPLAY ALL-SAMPLE RESULTS
# ============================================================

print("\n" + "=" * 90)
print("ALL-SAMPLE PERFORMANCE")
print("=" * 90)

all_sample_columns = [
    "antibiotic",
    "n_samples",
    "n_resistant",
    "n_susceptible",
    "balanced_accuracy",
    "recall_R",
    "recall_S",
    "f1_R",
    "f1_S",
    "f1_macro",
    "auroc",
    "pr_auc",
    "false_susceptible",
    "false_resistant",
]

display(
    all_sample_results[
        all_sample_columns
    ]
    .sort_values(
        "balanced_accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
    .round(4)
)

# ============================================================
# DISPLAY HIGH-CONFIDENCE CALLED-ONLY RESULTS
# ============================================================

print("\n" + "=" * 90)
print("HIGH-CONFIDENCE CALLED-ONLY PERFORMANCE")
print("=" * 90)

if len(called_only_results) > 0:

    called_only_columns = [
        "antibiotic",
        "n_samples",
        "n_resistant",
        "n_susceptible",
        "coverage",
        "no_call_rate",
        "balanced_accuracy",
        "recall_R",
        "recall_S",
        "f1_R",
        "f1_S",
        "f1_macro",
        "auroc",
        "pr_auc",
        "false_susceptible",
        "false_resistant",
    ]

    display(
        called_only_results[
            called_only_columns
        ]
        .sort_values(
            "balanced_accuracy",
            ascending=False,
        )
        .reset_index(drop=True)
        .round(4)
    )

else:

    print(
        "No called-only results were generated."
    )

# ============================================================
# COMPARE ALL-SAMPLE VS CALLED-ONLY PERFORMANCE
# ============================================================

if len(called_only_results) > 0:

    comparison = (
        all_sample_results[
            [
                "antibiotic",
                "balanced_accuracy",
                "recall_R",
                "recall_S",
                "f1_macro",
                "auroc",
                "pr_auc",
            ]
        ]
        .rename(
            columns={
                "balanced_accuracy":
                    "all_balanced_accuracy",

                "recall_R":
                    "all_recall_R",

                "recall_S":
                    "all_recall_S",

                "f1_macro":
                    "all_f1_macro",

                "auroc":
                    "all_auroc",

                "pr_auc":
                    "all_pr_auc",
            }
        )
        .merge(
            called_only_results[
                [
                    "antibiotic",
                    "coverage",
                    "no_call_rate",
                    "balanced_accuracy",
                    "recall_R",
                    "recall_S",
                    "f1_macro",
                    "auroc",
                    "pr_auc",
                ]
            ]
            .rename(
                columns={
                    "balanced_accuracy":
                        "called_balanced_accuracy",

                    "recall_R":
                        "called_recall_R",

                    "recall_S":
                        "called_recall_S",

                    "f1_macro":
                        "called_f1_macro",

                    "auroc":
                        "called_auroc",

                    "pr_auc":
                        "called_pr_auc",
                }
            ),
            on="antibiotic",
            how="left",
        )
    )

    comparison[
        "balanced_accuracy_gain"
    ] = (
        comparison[
            "called_balanced_accuracy"
        ]
        - comparison[
            "all_balanced_accuracy"
        ]
    )

    comparison[
        "recall_R_gain"
    ] = (
        comparison[
            "called_recall_R"
        ]
        - comparison[
            "all_recall_R"
        ]
    )

    comparison[
        "recall_S_gain"
    ] = (
        comparison[
            "called_recall_S"
        ]
        - comparison[
            "all_recall_S"
        ]
    )

    print("\n" + "=" * 90)
    print("ALL-SAMPLE VS HIGH-CONFIDENCE COMPARISON")
    print("=" * 90)

    display(
        comparison.round(4)
    )

# ============================================================
# MEAN METRICS ACROSS ANTIBIOTICS
# ============================================================

print("\n" + "=" * 90)
print("MEAN ALL-SAMPLE METRICS")
print("=" * 90)

mean_all_metrics = (
    all_sample_results[
        [
            "balanced_accuracy",
            "recall_R",
            "recall_S",
            "f1_R",
            "f1_S",
            "f1_macro",
            "auroc",
            "pr_auc",
        ]
    ]
    .mean()
    .to_frame(
        name="mean"
    )
)

display(
    mean_all_metrics.round(4)
)

if len(called_only_results) > 0:

    print("\n" + "=" * 90)
    print("MEAN HIGH-CONFIDENCE METRICS")
    print("=" * 90)

    mean_called_metrics = (
        called_only_results[
            [
                "coverage",
                "no_call_rate",
                "balanced_accuracy",
                "recall_R",
                "recall_S",
                "f1_R",
                "f1_S",
                "f1_macro",
                "auroc",
                "pr_auc",
            ]
        ]
        .mean()
        .to_frame(
            name="mean"
        )
    )

    display(
        mean_called_metrics.round(4)
    )

# ============================================================
# FINAL PATH REPORT
# ============================================================

print("\nSaved full evaluation:")
print(EVALUATION_PATH)

print("\nSaved all-sample evaluation:")
print(ALL_SAMPLE_RESULTS_PATH)

print("\nSaved called-only evaluation:")
print(CALLED_ONLY_RESULTS_PATH)

MODULE 05: MODEL EVALUATION

Prediction file:
/content/drive/MyDrive/genome-firewall/data/predictions/test_predictions.csv

Prediction rows:
1104

Prediction columns:
['genome_id', 'antibiotic', 'y_true', 'y_prob', 'is_called', 'predicted_label', 'confident_prediction', 'prediction_text', 'confidence', 'confidence_margin', 'model', 'split']

Antibiotics:
['ampicillin', 'cefotaxime', 'ciprofloxacin', 'gentamicin', 'trimethoprim/sulfamethoxazole']

Prediction preview:


,genome_id,antibiotic,y_true,y_prob,is_called,predicted_label,confident_prediction,prediction_text,confidence,confidence_margin,model,split
0,562.100005,ampicillin,1,0.079164,True,0,0.0,Susceptible,0.920836,0.420836,logistic_regression,test
1,562.100116,ampicillin,1,0.999993,True,1,1.0,Resistant,0.999993,0.499993,logistic_regression,test
2,562.100135,ampicillin,1,0.986806,True,1,1.0,Resistant,0.986806,0.486806,logistic_regression,test
3,562.100136,ampicillin,1,0.977417,True,1,1.0,Resistant,0.977417,0.477417,logistic_regression,test
4,562.100170,ampicillin,1,0.992314,True,1,1.0,Resistant,0.992314,0.492314,logistic_regression,test



Overall confident calls:
995 / 1104

Overall coverage:
0.9013

Overall no-call rate:
0.0987

FULL HELD-OUT TEST RESULTS


,antibiotic,evaluation_scope,n_samples,n_resistant,n_susceptible,coverage,no_call_rate,balanced_accuracy,recall_R,recall_S,f1_R,f1_S,f1_macro,auroc,pr_auc,threshold,true_resistant,true_susceptible,false_resistant,false_susceptible
0,ampicillin,all_samples,216,127,89,1.0000,0.0000,0.9488,0.8976,1.0000,0.9461,0.9319,0.9390,0.9639,0.9820,0.5,114,89,0,13
1,ampicillin,called_only,207,119,88,0.9583,0.0417,0.9580,0.9160,1.0000,0.9561,0.9462,0.9512,0.9620,0.9803,0.5,109,88,0,10
2,cefotaxime,all_samples,231,48,183,1.0000,0.0000,0.9216,0.8542,0.9891,0.9011,0.9757,0.9384,0.9400,0.9276,0.5,41,181,2,7
3,cefotaxime,called_only,224,45,179,0.9697,0.0303,0.9444,0.8889,1.0000,0.9412,0.9862,0.9637,0.9383,0.9275,0.5,40,179,0,5
4,ciprofloxacin,all_samples,261,67,194,1.0000,0.0000,0.9676,0.9403,0.9948,0.9618,0.9872,0.9745,0.9927,0.9856,0.5,63,193,1,4
5,ciprofloxacin,called_only,241,62,179,0.9234,0.0766,0.9839,0.9677,1.0000,0.9836,0.9944,0.9890,0.9958,0.9911,0.5,60,179,0,2
6,gentamicin,all_samples,258,28,230,1.0000,0.0000,0.9474,0.9643,0.9304,0.7606,0.9618,0.8612,0.9632,0.8960,0.5,27,214,16,1
7,gentamicin,called_only,197,15,182,0.7636,0.2364,0.9639,0.9333,0.9945,0.9333,0.9945,0.9639,0.9440,0.8912,0.5,14,181,1,1
8,trimethoprim/sulfamethoxazole,all_samples,138,50,88,1.0000,0.0000,0.9373,0.9200,0.9545,0.9200,0.9545,0.9373,0.9459,0.8969,0.5,46,84,4,4
9,trimethoprim/sulfamethoxazole,called_only,126,48,78,0.9130,0.0870,0.9391,0.9167,0.9615,0.9263,0.9554,0.9409,0.9487,0.9015,0.5,44,75,3,4



ALL-SAMPLE PERFORMANCE


...


MEAN ALL-SAMPLE METRICS


,mean
balanced_accuracy,0.9445
recall_R,0.9153
recall_S,0.9738
f1_R,0.8979
f1_S,0.9622
f1_macro,0.9301
auroc,0.9611
pr_auc,0.9376



MEAN HIGH-CONFIDENCE METRICS


,mean
coverage,0.9056
no_call_rate,0.0944
balanced_accuracy,0.9579
recall_R,0.9245
recall_S,0.9912
f1_R,0.9481
f1_S,0.9754
f1_macro,0.9617
auroc,0.9577
pr_auc,0.9383



Saved full evaluation:
/content/drive/MyDrive/genome-firewall/results/evaluation_by_antibiotic.csv

Saved all-sample evaluation:
/content/drive/MyDrive/genome-firewall/results/evaluation_all_samples.csv

Saved called-only evaluation:
/content/drive/MyDrive/genome-firewall/results/evaluation_called_only.csv
